# Classical AI Reasoning Agent

**Roll number: M26CSA002 | AI Assignment 2026**

This notebook contains the complete implementation and offline public evaluation snapshots.
Run all cells in order with Python 3.10 or newer. No API key, local source folder,
download, or third-party Python package is required for inference or evaluation.

The system combines indexed Horn inference, lifted unit resolution, an independently
implemented DPLL engine for bounded first-order grounding, and Boolean causal-model
enumeration. The default language parser is deterministic. An optional local Qwen
model translates English to formulas; it never supplies the final truth label.
Its live use requires the separate model/runtime download described below. The final prediction is always
computed by these classical algorithms; public labels are used only for scoring.

Default entailment semantics are open-world with explicit negative facts. Optional
stratified CWA is evaluated separately on the alternative RuleTaker data. FOLIO's
False and Uncertain both map to False for binary entailment; the 134-example
True/False-only metric is also printed for a like-for-like baseline comparison.

Full formal-annotation FOLIO evaluation is separate from the required language-only
pipeline. SAT after bounded grounding means **not proved**, not a complete FOL
countermodel. Unsupported syntax/resource limits are reported and map to False.


## Runtime imports and error types

In [1]:
"""Classical reasoning agent. Standard library only; no learned answer predictor.

Three paths: indexed Horn closure, bounded first-order ground refutation using
our DPLL solver, and exact enumeration of Boolean structural causal models.
Unsupported syntax and resource limits are surfaced in Result.diagnostics.
"""
from __future__ import annotations
import csv
import itertools
import json
import re
import time
from collections import defaultdict, OrderedDict, Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional




## Formula representation and parsing

In [2]:
class ParseError(ValueError):
    pass

class ResourceLimit(RuntimeError):
    pass

def atom(name, *args):
    return ('atom', name.lower(), tuple(args))

def neg(f):
    return f[1] if f[0] == 'not' else ('not', f)

def junction(op, *items):
    out = []
    for f in items:
        out.extend(f[1:] if f[0] == op else [f])
    return out[0] if len(out) == 1 else (op, *out)

def variable(t):
    return isinstance(t, str) and t.startswith('?')

def term_vars(t):
    if variable(t): return {t}
    if isinstance(t, tuple): return set().union(*(term_vars(v) for v in t[2]))
    return set()

def term_sub(t, sub):
    if variable(t):
        seen = set()
        while variable(t) and t in sub and t not in seen:
            seen.add(t); t = sub[t]
        return term_sub(t, sub) if isinstance(t, tuple) else t
    if isinstance(t, tuple): return ('fn', t[1], tuple(term_sub(a, sub) for a in t[2]))
    return t

def formula_sub(f, sub):
    if f[0] == 'atom': return ('atom', f[1], tuple(term_sub(t, sub) for t in f[2]))
    if f[0] in ('all','exists'):
        return (f[0], f[1], formula_sub(f[2], {k:v for k,v in sub.items() if k != f[1]}))
    return (f[0], *(formula_sub(x, sub) for x in f[1:]))

def show(f):
    if f[0] == 'atom': return f[1] + '(' + ','.join(map(str,f[2])) + ')'
    if f[0] == 'not': return 'not ' + show(f[1])
    if f[0] in ('all','exists'): return f'{f[0]} {f[1]} ({show(f[2])})'
    return '(' + (' '+f[0]+' ').join(show(x) for x in f[1:]) + ')'

class FormulaParser:
    """Recursive-descent parser preserving quantifier scope and full queries."""
    token_re = re.compile(r'\s*(<->|->|!=|[∀∃¬∧∨→↔⊕⊻=(),~&|!]|[\w][\w.-]*)')
    def parse(self, text):
        text = text.strip().rstrip('.').replace('⇒','→').replace('⇔','↔').replace('⟷','↔')
        self.tokens=[];pos=0
        while pos < len(text):
            m=self.token_re.match(text,pos)
            if not m: raise ParseError('Unrecognized FOL token: '+text[pos:pos+30])
            self.tokens.append(m.group(1));pos=m.end()
        self.i=0;self.bound={};self.serial=0
        if not self.tokens: raise ParseError('Empty formula')
        f=self.binary(0)
        if self.i != len(self.tokens): raise ParseError('Trailing FOL tokens: '+' '.join(self.tokens[self.i:]))
        return f
    def peek(self): return self.tokens[self.i] if self.i<len(self.tokens) else ''
    def take(self,expected=None):
        t=self.peek()
        if not t or (expected is not None and t != expected): raise ParseError(f'Expected {expected}, found {t}')
        self.i+=1;return t
    def binary(self,level):
        ops=[{'↔':'iff','<->':'iff'},{'→':'implies','->':'implies'},{'⊕':'xor','⊻':'xor'},{'∨':'or','|':'or'},{'∧':'and','&':'and'}]
        if level == len(ops): return self.unary()
        left=self.binary(level+1)
        while self.peek() in ops[level]:
            op=ops[level][self.take()]
            right=self.binary(level if op=='implies' else level+1)
            left=junction(op,left,right) if op in ('and','or') else (op,left,right)
        return left
    def unary(self):
        t=self.peek()
        if t in ('¬','~','!'):
            self.take();return neg(self.unary())
        if t in ('∀','∃'):
            q='all' if self.take()=='∀' else 'exists'
            original=self.take();self.serial+=1;v=f'?{original}_{self.serial}'
            prev=self.bound.get(original);self.bound[original]=v
            if self.peek() in (',','.'):self.take()
            body=self.binary(0)
            if prev is None:self.bound.pop(original)
            else:self.bound[original]=prev
            return (q,v,body)
        if t=='(':
            self.take();f=self.binary(0);self.take(')');return f
        name=self.take()
        if not re.fullmatch(r'[\w.-]+',name):raise ParseError('Invalid atom '+name)
        if self.peek()=='(':
            self.take();args=[]
            if self.peek()!=')':
                args.append(self.term())
                while self.peek()==',':self.take();args.append(self.term())
            self.take(')')
            if self.peek() in ('=','!='):
                op=self.take();f=atom('=',('fn',name.lower(),tuple(args)),self.term())
                return neg(f) if op=='!=' else f
            return atom(name,*args)
        left=self.bound.get(name,name.lower())
        if self.peek() in ('=','!='):
            op=self.take();f=atom('=',left,self.term());return neg(f) if op=='!=' else f
        return atom(name)
    def term(self):
        name=self.take()
        if not re.fullmatch(r'[\w.-]+',name):raise ParseError('Invalid term '+name)
        if self.peek()=='(':
            self.take();args=[self.term()]
            while self.peek()==',':self.take();args.append(self.term())
            self.take(')');return ('fn',name.lower(),tuple(args))
        return self.bound.get(name,name.lower())




## Own classical refutation solver

In [3]:
def nnf(f, negative=False):
    op=f[0]
    if op=='atom': return ('not',f) if negative else f
    if op=='not': return nnf(f[1],not negative)
    if op=='implies': return nnf(junction('or',neg(f[1]),f[2]),negative)
    if op=='iff':
        return nnf(junction('and',('implies',f[1],f[2]),('implies',f[2],f[1])),negative)
    if op=='xor':
        return nnf(junction('and',junction('or',f[1],f[2]),junction('or',neg(f[1]),neg(f[2]))),negative)
    if op in ('all','exists'):
        q=({'all':'exists','exists':'all'}[op] if negative else op)
        return (q,f[1],nnf(f[2],negative))
    if op in ('and','or'):
        return junction(({'and':'or','or':'and'}[op] if negative else op),*(nnf(x,negative) for x in f[1:]))
    raise ParseError('Unknown formula operator '+op)

def clausify(formulas):
    """Standardize bound variables, Skolemize, distribute to CNF; no shortcuts."""
    seq=itertools.count();functions={}
    def skolem(f,universals=(),sub=None):
        sub={} if sub is None else sub
        if f[0] in ('all','exists'):
            key=f[1];sub=dict(sub);n=next(seq)
            if f[0]=='all':
                v=f'?u{n}';sub[key]=v
                return skolem(f[2],universals+(v,),sub)
            name=f'$sk{n}'
            term=('fn',name,universals) if universals else name
            if universals:functions[name]=len(universals)
            sub[key]=term
            return skolem(f[2],universals,sub)
        if f[0]=='atom':return formula_sub(f,sub)
        return (f[0],*(skolem(x,universals,sub) for x in f[1:]))
    def cnf(f):
        if f[0]=='atom':return [frozenset([(True,f)])]
        if f[0]=='not':return [frozenset([(False,f[1])])]
        if f[0]=='and':return [c for x in f[1:] for c in cnf(x)]
        if f[0]=='or':
            acc=[frozenset()]
            for x in f[1:]:
                right=cnf(x)
                if len(acc)*len(right)>30000:raise ResourceLimit('CNF distribution limit')
                acc=[a|b for a in acc for b in right]
            return acc
        raise ParseError('CNF conversion failed')
    # Fresh definitional predicates avoid exponential expansion of nested
    # XOR/iff. Each definition is universally closed over its free variables,
    # so it remains an equisatisfiable conservative extension in FOL.
    definitions=[];definition_serial=itertools.count()
    def abbreviate(f):
        if f[0]=='atom':return f
        if f[0] in ('all','exists'):return (f[0],f[1],abbreviate(f[2]))
        out=(f[0],*(abbreviate(x) for x in f[1:]))
        if f[0] in ('xor','iff'):
            variables=sorted(formula_free_vars(out))
            name=atom('$definition'+str(next(definition_serial)),*variables)
            definition=('iff',name,out)
            for v in reversed(variables):definition=('all',v,definition)
            definitions.append(definition)
            return name
        return out
    formulas=[abbreviate(f) for f in formulas]+definitions
    clauses=[]
    for f in formulas:
        clauses.extend(cnf(skolem(nnf(f))))
    return clauses,functions

def unify_term(a,b,sub):
    a=term_sub(a,sub);b=term_sub(b,sub)
    if a==b:return sub
    if variable(a):
        if a in term_vars(b):return None
        sub[a]=b;return sub
    if variable(b):return unify_term(b,a,sub)
    if isinstance(a,tuple) and isinstance(b,tuple) and a[:2]==b[:2] and len(a[2])==len(b[2]):
        for x,y in zip(a[2],b[2]):
            if unify_term(x,y,sub) is None:return None
        return sub
    return None

def unify_atoms(a,b,sub=None):
    if a[1]!=b[1] or len(a[2])!=len(b[2]):return None
    sub=dict(sub or {})
    for x,y in zip(a[2],b[2]):
        if unify_term(x,y,sub) is None:return None
    return sub

def dpll(clauses, deadline, stats):
    """Davis-Putnam-Logemann-Loveland: unit propagation and branching.

    True = satisfiable; False = a refutation, with no third-party solver.
    """
    stats['nodes']+=1
    if stats['nodes']%64==1 and time.perf_counter()>deadline:raise ResourceLimit('DPLL time limit')
    while True:
        if any(not c for c in clauses):return False
        if not clauses:return True
        units={next(iter(c)) for c in clauses if len(c)==1}
        if any(-u in units for u in units):return False
        if not units:break
        stats['units']+=len(units)
        clauses=[c-{-u for u in units} for c in clauses if not c & units]
    # Pure-literal elimination preserves satisfiability.
    literals=set().union(*clauses)
    pure={v for v in literals if -v not in literals}
    if pure:
        clauses=[c for c in clauses if not c & pure]
        if not clauses:return True
    shortest=min(clauses,key=len)
    counts=Counter(x for c in clauses for x in c)
    chosen=max(shortest,key=lambda x:counts[x]+counts[-x])
    for lit in (chosen,-chosen):
        reduced=[c-{-lit} for c in clauses if lit not in c]
        if dpll(reduced,deadline,stats):return True
    return False

def unit_refutation(clauses,deadline,max_clauses=2500):
    """Sound lifted unit resolution before grounding; no SAT/model claim."""
    pool=[];seen=set();index=defaultdict(set);units=[];pairs=set();serial=itertools.count()
    def normalize(clause):
        rename={}
        for _,a in sorted(clause,key=repr):
            for v in sorted(formula_free_vars(a)):
                if v not in rename:rename[v]='?v'+str(len(rename))
        return frozenset((s,formula_sub(a,rename)) for s,a in clause)
    def add(clause):
        clause=normalize(clause)
        if any((not s,a) in clause for s,a in clause) or clause in seen:return
        seen.add(clause);i=len(pool);pool.append(clause)
        for s,a in clause:index[(s,a[1],len(a[2]))].add(i)
        if len(clause)==1:units.append(i)
    for clause in clauses:add(clause)
    add(frozenset([(True,atom('=','?reflexive','?reflexive'))]))
    iterations=0;cursor=0
    while cursor<len(units) and len(pool)<max_clauses and time.perf_counter()<deadline:
        ui=units[cursor];cursor+=1
        s,a=next(iter(pool[ui]))
        for ci in list(index[(not s,a[1],len(a[2]))]):
            if (ui,ci) in pairs:continue
            pairs.add((ui,ci));iterations+=1
            if iterations%64==0 and time.perf_counter()>deadline:break
            clause=pool[ci];tag=str(next(serial))
            vs=set().union(*(formula_free_vars(b) for _,b in clause))
            clause=frozenset((sign,formula_sub(b,{v:f'?r{tag}_{j}' for j,v in enumerate(sorted(vs))})) for sign,b in clause)
            for sign,b in clause:
                if sign==s:continue
                substitution=unify_atoms(a,b)
                if substitution is None:continue
                result=frozenset((pol,formula_sub(c,substitution)) for pol,c in clause if (pol,c)!=(sign,b))
                if not result:return True,iterations,len(pool)
                old_length=len(pool);add(result)
                if len(pool)>old_length and len(result)>1:
                    # Existing units must also visit the newly derived clause.
                    for existing in units[:cursor]:
                        es,ea=next(iter(pool[existing]))
                        if any(ps!=es and pa[1:2]==ea[1:2] and len(pa[2])==len(ea[2]) for ps,pa in result):
                            units.append(existing)
    return any(not c for c in pool),iterations,len(pool)


def prove_fol(premises,query,timeout=1.0,max_ground=50000):
    """Sound, bounded refutation over ground instances of Skolemized clauses.

    UNSAT proves entailment in FOL. SAT is only 'not proved' (no domain-closure
    or unique-name assumption). Bounded term depth can miss valid proofs.
    """
    start=time.perf_counter();deadline=start+timeout
    stats={'nodes':0,'units':0,'ground_clauses':0};trace=[]
    clauses,functions=clausify(list(premises)+[neg(query)])
    refuted,resolutions,derived=unit_refutation(clauses,min(deadline,start+min(0.06,timeout/5)))
    stats.update(lifted_resolutions=resolutions,lifted_clauses=derived)
    if refuted:
        trace.append(f'Lifted unit resolution refuted KB and negated query ({resolutions} unifications, {derived} clauses).')
        return True,trace,stats
    constants=set();seen_functions=dict(functions)
    def collect(t):
        if isinstance(t,tuple):
            seen_functions[t[1]]=len(t[2])
            for x in t[2]:collect(x)
        elif not variable(t):constants.add(t)
    for c in clauses:
        for _,a in c:
            for t in a[2]:collect(t)
    constants=constants or {'$domain0'}
    terms=set(constants);ground=set()
    # More terms are an added proof resource, never a finite-domain axiom.
    for depth in range(2 if seen_functions else 1):
        if depth:
            expanded=set(terms)
            for name,arity in seen_functions.items():
                if len(terms)**arity>1500:continue
                for xs in itertools.product(sorted(terms,key=repr),repeat=arity):
                    expanded.add(('fn',name,xs))
            terms=expanded
        for c in clauses:
            vs=sorted(set().union(*(term_vars(t) for _,a in c for t in a[2])))
            if len(terms)**len(vs)>max_ground:raise ResourceLimit('Grounding budget exceeded')
            for xs in itertools.product(sorted(terms,key=repr),repeat=len(vs)):
                sub=dict(zip(vs,xs));g=frozenset((s,formula_sub(a,sub)) for s,a in c)
                if any((not s,a) in g for s,a in g):continue
                ground.add(g)
                if len(ground)>max_ground:raise ResourceLimit('Ground clause limit')
            if time.perf_counter()>deadline:raise ResourceLimit('Grounding time limit')
        # Equality is a congruence, not unique names. Ground reflexivity,
        # symmetry, transitivity and predicate/function congruence are valid
        # FOL axioms; adding any finite subset preserves soundness of UNSAT.
        atoms={a for c in ground for _,a in c}
        if any(a[1]=='=' for a in atoms):
            relevant=set(t for a in atoms for t in a[2])
            if len(relevant)>35:raise ResourceLimit('Equality term budget')
            ts=sorted(relevant,key=repr)
            for a in ts:ground.add(frozenset([(True,atom('=',a,a))]))
            for a,b in itertools.product(ts,repeat=2):
                ground.add(frozenset([(False,atom('=',a,b)),(True,atom('=',b,a))]))
            for a,b,c in itertools.product(ts,repeat=3):
                ground.add(frozenset([(False,atom('=',a,b)),(False,atom('=',b,c)),(True,atom('=',a,c))]))
            by_pred=defaultdict(list)
            for a in atoms:
                if a[1]!='=':by_pred[(a[1],len(a[2]))].append(a)
            for group in by_pred.values():
                for a,b in itertools.combinations(group,2):
                    eqs=[(False,atom('=',x,y)) for x,y in zip(a[2],b[2]) if x!=y]
                    ground.add(frozenset(eqs+[(False,a),(True,b)]))
                    ground.add(frozenset(eqs+[(True,a),(False,b)]))
            fs=[x for x in ts if isinstance(x,tuple)]
            for a,b in itertools.combinations(fs,2):
                if a[:2]==b[:2] and len(a[2])==len(b[2]):
                    ground.add(frozenset([(False,atom('=',x,y)) for x,y in zip(a[2],b[2])]+[(True,atom('=',a,b))]))
        ids={};encoded=[]
        for c in sorted(ground,key=lambda c:(len(c),repr(sorted(c,key=repr)))):
            row=set()
            for sign,a in c:
                if a not in ids:ids[a]=len(ids)+1
                row.add(ids[a] if sign else -ids[a])
            encoded.append(frozenset(row))
        stats['ground_clauses']=len(encoded)
        satisfiable=dpll(encoded,deadline,stats)
        trace.append(f'Ground term depth {depth}: {len(ids)} atoms, {len(encoded)} clauses; '+('SAT (proof not found)' if satisfiable else 'UNSAT'))
        if not satisfiable:
            trace.append('KB and negated query refuted by DPLL; query is entailed.')
            return True,trace,stats
    return False,trace,stats




## Natural-language parsing

In [4]:
def slug(s):
    return re.sub(r'[^a-z0-9]+','_',s.lower()).strip('_')

def singular(w):
    irregular={'people':'person','men':'man','women':'woman','children':'child','mice':'mouse','geese':'goose','teeth':'tooth','movies':'movie','cookies':'cookie','pies':'pie','ties':'tie','series':'series','species':'species'}
    if w in irregular:return irregular[w]
    if w.endswith('ies') and len(w)>3:return w[:-3]+'y'
    if w.endswith(('sses','shes','ches','xes','zes')):return w[:-2]
    if w.endswith('s') and not w.endswith(('ss','us','is')):return w[:-1]
    return w

def lemma(w):
    table={'is':'be','are':'be','was':'be','were':'be','has':'have','does':'do',
           'sees':'see','chases':'chase','likes':'like','loves':'love','hates':'hate',
           'uses':'use','becomes':'become','dances':'dance','creates':'create','takes':'take',
           'writes':'write','gives':'give','goes':'go','studies':'study','knows':'know',
           'ate':'eat','saw':'see','had':'have','went':'go','knew':'know','liked':'like',
           'loved':'love','visited':'visit','attended':'attend','supported':'support',
           'published':'publish','worked':'work','played':'play','lived':'live',
           'makes':'make','made':'make','spends':'spend','spent':'spend',
           'gets':'get','got':'get','sits':'sit','sat':'sit','contains':'contain',
           'exercises':'exercise','produces':'produce','receives':'receive',
           'drives':'drive','chooses':'choose','rents':'rent','fails':'fail',
           'passes':'pass','passed':'pass','failed':'fail','allows':'allow','sends':'send','starts':'start',
           'wakes':'wake','begins':'begin','believes':'believe','belongs':'belong',
           'lost':'lose','left':'leave','sold':'sell','wrote':'write','led':'lead',
           'leads':'lead','created':'create','moved':'move','hosted':'host',
           'adores':'adore','schedules':'schedule','performs':'perform',
           'wishes':'wish','chaperones':'chaperone','includes':'include',
           'grew':'grow','fulfills':'fulfill','invites':'invite','holds':'hold',
           'leaps':'leap','activates':'activate','disregards':'disregard'}
    return table.get(w,singular(w))

def clean_nl(s):
    s=s.replace('\u2019',"'").replace('\u2018',"'")
    s=re.sub(r"\b(isn't|aren't|wasn't|weren't)\b",lambda m:{"isn't":"is not","aren't":"are not","wasn't":"was not","weren't":"were not"}[m[0].lower()],s,flags=re.I)
    s=re.sub(r"\b(doesn't|don't|didn't|cannot|can't|won't)\b",lambda m:{"doesn't":"does not","don't":"do not","didn't":"did not","cannot":"can not","can't":"can not","won't":"will not"}[m[0].lower()],s,flags=re.I)
    return re.sub(r'\s+',' ',s).strip(' .?!;,').lower()

class NLParser:
    """Deterministic compositional parser. Vocabulary is not a label lookup.

    Recognized rules include if/then, assuming, given that, based on, when,
    universal/no/some statements, conjunction, disjunction and relative clauses.
    Unrecognized constructions are reported instead of discarded conditions.
    """
    verb_words=set('is are was were likes like chases chase sees see eats eat helps help visits visit knows know loves love hates hate needs need wants want has have owns own becomes become dances dance creates create fishes fish volunteers volunteer runs run grows grow travels travel hikes hike explores explore writes write takes take rides ride sings sing paints paint collects collect hosts host enjoys enjoy plays play attends attend works work lives live drinks drink supports support supported published attended loved can cannot could will does do did goes go teaches teach studies study speaks speak reads read swims swim gives give buys buy sells sell admires admire respects respect parent'.split())
    pronouns={'someone','something','somebody','anyone','anything','everyone','everything','everybody','they','them','he','she','it','themselves','him','her','that person','that thing'}
    verb_words.update({'use','uses','cook','cooks','prefer','prefers','subscribe','subscribes','watch','watches','communicate','communicates'})
    verb_words.update('make makes made spend spends spent get gets got sit sits sat contain contains exercise exercises produce produces receive receives drive drives choose chooses rent rents fail fails failed pass passes passed allow allows send sends start starts wake wakes begin begins believe believes belong belongs specialize specialized sting stings control controls borrow borrows need needs must should'.split())
    verb_words.update('schedule schedules perform performs performed wish wishes chaperone chaperones adore adores lose lost leave leaves left sold wrote lead leads led include includes included move moves moved receive received create created contain contained appear appears provide provides reproduce reproduces complain complains spill spills value values prioritize prioritizes gain gains read reads win wins fight fights invite invites fulfill fulfills grew disregard disregards activate activates leap leaps hold holds register registers participate participates'.split())
    adverbs={'regularly','sometimes','never','always','frequently','often'}
    def __init__(self):
        self.entities=set();self.diagnostics=[];self.classical=False;self.query_mode=False;self.nominals=set()
    def term(self,s,role='subject',env=None):
        env={} if env is None else env
        s=clean_nl(s)
        if s=='it' and 'object' in env:return env['object']
        if s in env:return env[s]
        if s in self.pronouns:
            if role=='object' and s in ('something','someone','somebody','anything','anyone'):
                env.setdefault('object','?y');env[s]=env['object'];return env['object']
            if role=='object' and s in ('it','them','him','her'):
                return env.get('object',env.get('subject','?x'))
            return env.get('subject','?x')
        s=re.sub(r'^(?:the|a|an)\s+','',s)
        if not s:raise ParseError('Missing entity')
        if role=='object' and slug(s) not in getattr(self,'proper_entities',set()):
            s=re.sub(r'\b(?:the|a|an)\s+','',s)
            s=re.sub(r'\b(?:his|her|its|their)\b','their',s)
            s=' '.join(singular(w) for w in s.split())
        self.entities.add(slug(s));return slug(s)
    def subject_vp(self,text):
        text=clean_nl(text)
        # Resolve a known name before inspecting verbs inside its predicate.
        for ent in sorted(self.entities,key=len,reverse=True):
            for name in (ent.replace('_',' '),'the '+ent.replace('_',' ')):
                if text.startswith(name+' '):
                    rest=text[len(name)+1:]
                    if rest.split()[0] in self.verb_words|self.adverbs|{'either','neither','both'}:
                        return name,rest
        words=text.split()
        for i in range(1,len(words)):
            if words[i] in self.verb_words or words[i] in ('either','neither') or words[i] in self.adverbs:
                start=i-1 if words[i-1]=='both' else i
                return ' '.join(words[:start]),' '.join(words[start:])
        if len(words)>=2:
            pos=2 if words[0] in ('the','a','an') and len(words)>=3 else 1
            return ' '.join(words[:pos]),' '.join(words[pos:])
        raise ParseError('No predicate in '+text)
    def property(self,phrase,subject,env):
        phrase=clean_nl(phrase)
        negative=False
        if phrase.startswith('not '):negative=True;phrase=phrase[4:]
        phrase=re.sub(r'^(?:also\s+)?(?:a\s+|an\s+|the\s+)?','',phrase)
        phrase=re.sub(r'\s+(?:things?|ones?|persons?|people)$','',phrase)
        m=re.match(r'^with\s+(no\s+)?(.+)$',phrase)
        if m:
            f=self.vp(('does not have ' if m[1] else 'has ')+m[2],subject,env)
            return neg(f) if negative else f
        m=re.match(r'^(.+?)\s+(working|taking|having|containing|displayed|located)\s+(.+)$',phrase)
        if m:
            f=junction('and',self.property(m[1],subject,env),self.vp('is '+m[2]+' '+m[3],subject,env))
            return neg(f) if negative else f
        # Relative clauses and noun modifiers stay separate conjuncts. This
        # allows the same type/property to be recognized in another sentence.
        m=re.match(r'^(.+?)\s+(?:who|that|which)\s+(.+)$',phrase)
        if m and m[1] not in ('aware','unaware','certain','uncertain','sure','true','false'):
            f=junction('and',self.property(m[1],subject,env),self.vp(m[2],subject,env))
            return neg(f) if negative else f
        phrase=re.sub(r'^located\s+(?=in\b|at\b)','',phrase)
        m=re.match(r'^(in|at|on|from|with)\s+(.+)$',phrase)
        if m:
            f=atom(m[1],subject,self.term(m[2],'object',env))
            return neg(f) if negative else f
        m=re.match(r'^(.+?)\s+(in|at|from|with)\s+(.+)$',phrase)
        if m and m[1].split()[-1] not in self.verb_words and not m[1].split()[-1].endswith('ed') and m[1] not in ('good','bad','interested','fond'):
            f=junction('and',self.property(m[1],subject,env),self.property(m[2]+' '+m[3],subject,env))
            return neg(f) if negative else f
        # Relational copulas must preserve the second argument.
        m=re.match(r'^([a-z]+(?:\s+[a-z]+)?)\s+of\s+(.+)$',phrase)
        if m:f=atom(slug(' '.join(singular(w) for w in m[1].split()))+'_of',subject,self.term(m[2],'object',env))
        else:
            if not phrase:raise ParseError('Empty property')
            # Long relative descriptions are decomposed, rather than fused.
            m=re.match(r'^(.+?)\s+(?:who|that|which)\s+(.+)$',phrase)
            if m and m[1] not in ('aware','unaware','certain','uncertain','sure','true','false'):
                base=self.property(m[1],subject,env)
                f=junction('and',base,self.vp(m[2],subject,env))
            else:
                pred=' '.join(singular(w) for w in phrase.split())
                self.nominals.add(slug(pred))
                f=atom(slug(pred),subject)
        return neg(f) if negative else f
    def vp(self,text,subject,env):
        text=clean_nl(text)
        env.setdefault('subject',subject)
        text=re.sub(r'^(?:also|both)\s+','',text)
        text=re.sub(r'^currently\s+','',text)
        text=re.sub(r'^(is|are|was|were|does|do|did)\s+(he|she|it|they)\s+',r'\2 \1 ',text)
        if text in ('either','neither','both'):raise ParseError('Coordination has no antecedent: '+text)
        m=re.match(r'^(?:is|are|was|were)\s+(not\s+)?(\w+ing)\s*(.*)$',text)
        if m:
            stem=m[2][:-3]
            if stem+'e' in self.verb_words:stem+='e'
            if stem in self.verb_words:
                return self.vp(('does not ' if m[1] else '')+stem+' '+m[3],subject,env)
        m=re.match(r'^(?:has|have|had)\s+no\s+(.+)$',text)
        if m:return neg(self.vp('has '+m[1],subject,env))
        # Full coordinated clauses can restate a name or use a pronoun.
        if re.match(r'^(?:they|he|she|it)\s+',text):
            return self.expression(text,env=env)
        for ent in sorted(self.entities,key=len,reverse=True):
            for name in (ent.replace('_',' '),'the '+ent.replace('_',' ')):
                if text.startswith(name+' '):return self.expression(text,env=env)
        copula=re.match(r'^(is|are|was|were)\s+(.+)$',text)
        if copula and re.match(r'^(either|neither|both|not both|not either)\b',copula[2]):
            return self.coordination(copula[2],subject,env,'is ')
        if re.match(r'^(either|neither|not both|not either)\b',text):
            return self.coordination(text,subject,env)
        m=re.match(r'^(?:does|do|did)\s+not\s+(both|either)\s+(.+)$',text)
        if m:return neg(self.coordination(m[1]+' '+m[2],subject,env))
        if text.startswith('never '):return neg(self.vp(text[6:],subject,env))
        m=re.match(r'^(?:does|do|did)\s+not\s+(.+?\s+or\s+.+)$',text)
        if m and not re.search(r'\s+or\s+(?:does|do|did|is|are|he|she|it|they|not)\b',m[1]):
            return neg(self.vp(m[1],subject,env))
        # Share the same subject across elliptical conjunctions/disjunctions.
        if text.startswith('either '):
            text=text[7:]
            parts=re.split(r',?\s+or\s+',text,maxsplit=1)
            if len(parts)==2:return junction('or',self.vp(parts[0],subject,env),self.vp(parts[1],subject,env))
        if text.startswith('neither '):
            parts=re.split(r'\s+nor\s+',text[8:],maxsplit=1)
            if len(parts)==2:return junction('and',neg(self.vp(parts[0],subject,env)),neg(self.vp(parts[1],subject,env)))
        for sep,op in [(r',?\s+or\s+','or'),(r',?\s+and\s+','and')]:
            parts=re.split(sep,text)
            if len(parts)>1:
                # 'is smart and kind': later conjunct inherits the copula.
                fs=[]
                for index,part in enumerate(parts):
                    if index:part=self.complete_vp(part,parts[0])
                    fs.append(self.vp(part,subject,env))
                return junction(op,*fs)
        m=re.match(r'^(?:is|are|was|were)\s+(.+)$',text)
        if m:return self.property(m[1],subject,env)
        # A conjunction can contain a fully restated subject.
        if re.search(r'\s(?:is|are|was|were)\s',text):return self.expression(text,env=env)
        if text.startswith(('they ','it ','he ','she ')):
            return self.expression(text,env=env)
        negative=False
        m=re.match(r'^(?:does|do|did)\s+(not\s+)?(.+)$',text)
        if m:negative=bool(m[1]);text=m[2]
        elif text.startswith('not '):negative=True;text=text[4:]
        words=text.split()
        if not words:raise ParseError('Empty verb phrase')
        if words[0] in self.adverbs:
            modifier=words[0];rest=self.vp(' '.join(words[1:]),subject,env)
            if rest[0]=='atom':return atom(modifier+'_'+rest[1],*rest[2])
            return rest
        verb=lemma(words[0]);rest=' '.join(words[1:])
        if verb in ('can','could','will'):
            if rest.startswith('not '):negative=not negative;rest=rest[4:]
            words=rest.split();verb=verb+'_'+lemma(words[0]);rest=' '.join(words[1:])
        # Preserve a preposition in the predicate, e.g. works at / lives in.
        if rest.split()[:1] and rest.split()[0] in ('in','at','on','with','to','for','from','of'):
            prep,*tail=rest.split();verb+='_'+prep;rest=' '.join(tail)
        f=atom(slug(verb),subject,*([self.term(rest,'object',env)] if rest else []))
        return neg(f) if negative else f
    def complete_vp(self,fragment,previous,prefix=''):
        """Restore only a syntactically shared copula/verb, never a condition."""
        fragment=fragment.strip(' ,')
        first=fragment.split()[0] if fragment else ''
        if first in self.verb_words|self.adverbs|{'either','neither','they','he','she','it','both'}:return fragment
        if any(fragment.startswith(name+' ') for e in self.entities for name in (e.replace('_',' '),'the '+e.replace('_',' '))):return fragment
        if re.search(r'\s(?:is|are|was|were)\s',fragment):return fragment
        if prefix:return prefix+fragment
        m=re.match(r'^(?:is|are|was|were)\s+',previous)
        if m:return 'is '+fragment
        words=previous.split()
        if words and words[0] in ('does','do','did'):words=words[1:]
        if words and words[0]=='not':words=words[1:]
        if words:
            shared=words[0]
            if len(words)>1 and words[1] in ('in','at','on','with','to','for','from','of'):shared+=' '+words[1]
            return shared+' '+fragment
        return fragment
    def coordination(self,text,subject,env,prefix=''):
        self.classical=True
        if text.startswith('not both '):return neg(self.coordination('both '+text[9:],subject,env,prefix))
        if text.startswith('not either '):return neg(self.coordination('either '+text[11:],subject,env,prefix))
        negative=text.startswith('neither ')
        exclusive=bool(re.search(r',?\s+but not both$',text))
        text=re.sub(r',?\s+but not both$','',text)
        if negative:text=text[8:];parts=re.split(r',?\s+nor\s+',text);op='and'
        else:
            text=re.sub(r'^(?:either|both)\s+','',text)
            if re.search(r'\s+or\s+',text):parts=re.split(r',?\s+or\s+',text);op='xor' if exclusive else 'or'
            else:parts=re.split(r',?\s+and\s+',text);op='and'
        first=prefix+parts[0] if prefix and parts[0].split()[0] not in self.verb_words else parts[0]
        fs=[]
        for i,part in enumerate(parts):
            if i and re.fullmatch(r'(?:(?:he|she|it|they)\s+)?(?:(?:is|are|was|were|did|does|do)\s+)?(?:from\s+)?neither(?:\s+(?:is true|of these|of them))?',part.strip()):
                prior=fs[0];items=prior[1:] if prior[0]=='and' else [prior]
                fs.append(junction('and',*(neg(x) for x in items)));continue
            expanded=first if i==0 else self.complete_vp(part,first,prefix)
            f=self.vp(expanded,subject,env);fs.append(neg(f) if negative else f)
        return junction(op,*fs)
    def noun(self,text,v,env):
        text=clean_nl(text)
        text=re.sub(r'^of\s+(?:the\s+)?','',text)
        if text in ('thing','things','one','ones','person','people','object','objects','being','beings','someone','something','everyone','everything'):
            return ('and',)  # unrestricted domain
        m=re.match(r'^(.+?)\s+(?:who|that|which)\s+(.+)$',text)
        if m:return junction('and',self.noun(m[1],v,env),self.vp(m[2],v,env))
        m=re.match(r'^(?:people|persons?|things?|ones?|anyone|everyone)\s+(?:who|that)\s+(.+)$',text)
        if m:return self.vp(m[1],v,env)
        m=re.match(r'^(?:people|persons?|things?|ones?|anyone|everyone|everything)\s+(.+)$',text)
        if m and re.match(r'^(?:in|at|with|from|on|working|displayed|located)\b',m[1]):
            return self.property(m[1],v,env)
        m=re.match(r'^(.+?)\s+(displayed|located|made|managed|owned|working|containing)\s+(.+)$',text)
        if m:return junction('and',self.noun(m[1],v,env),self.vp('is '+m[2]+' '+m[3],v,env))
        text=re.sub(r'\s+(?:things?|people|persons?|ones?)$','',text)
        parts=re.split(r',\s*(?:and\s+)?|\s+and\s+',text)
        return junction('and',*(self.property(p,v,env) for p in parts if p))
    def quantified_parts(self,text):
        """Separate a restricted subject from its main finite verb phrase."""
        words=text.split();relative=next((i for i,w in enumerate(words) if w in ('who','that','which')),None)
        candidates=[]
        for i,w in enumerate(words):
            if i==0 or w not in self.verb_words:continue
            if words[i-1] in {'to','can','could','will','must','should','does','do','did','not'}:continue
            candidates.append(i)
        if relative is not None and candidates and candidates[0]>relative:
            candidates=[i for i in candidates if i>relative]
            # The first predicate belongs to the relative clause. A later
            # uncoordinated finite verb starts the main predicate.
            for i in candidates[1:]:
                if words[i-1] not in ('and','or','who','that','which','also'):
                    return ' '.join(words[:i]),' '.join(words[i:])
            raise ParseError('Cannot separate relative clause from main predicate: '+text)
        if candidates:
            i=candidates[0]
            if words[i-1] in self.adverbs or words[i-1]=='currently':i-=1
            return ' '.join(words[:i]),' '.join(words[i:])
        return self.subject_vp(text)
    def quantified(self,q,text,env):
        self.classical=True
        subject,vp=self.quantified_parts(text)
        v='?x';local={**env,'subject':v}
        restriction=self.noun(subject,v,local);head=self.vp(vp,v,local)
        if q=='no':head=neg(head)
        body=junction('and',restriction,head) if q=='some' else ('implies',restriction,head)
        return ('exists' if q=='some' else 'all',v,body)
    def expression(self,text,env=None,inherited=None):
        env={} if env is None else env
        text=clean_nl(text)
        text=re.sub(r',?\s+or both$','',text)
        if not text:raise ParseError('Empty statement')
        if text.startswith('either '):
            subj,_=self.subject_vp(text[7:]);subject=self.term(subj,'subject',env)
            return self.coordination(text,subject,{**env,'subject':subject})
        m=re.match(r'^there are (?:\d+|two|three|four|five|six|seven|eight|nine|ten) (?:types|kinds|categories) of (.+?):\s*(.+)$',text)
        if m:
            self.classical=True
            options=[x.strip() for x in re.split(r',\s*(?:and\s+)?|\s+and\s+',m[2]) if x.strip()]
            v='?x';local={**env,'subject':v}
            return ('all',v,('implies',self.noun(m[1],v,local),junction('or',*(self.noun(option,v,local) for option in options))))
        m=re.match(r'^(?:a\s+)?list of (.+?):\s*(.+)$',text)
        if m:
            names=[x.strip() for x in re.split(r',\s*(?:and\s+)?|\s+and\s+',m[2]) if x.strip()]
            return junction('and',*(self.noun(m[1],self.term(name),{}) for name in names))
        m=re.match(r'^there\s+(?:is|are|exists?|exist)\s+(?:at least one\s+|a\s+|an\s+|some\s+)?(.+)$',text)
        if m and ':' not in text:
            self.classical=True
            return ('exists','?x',self.noun(m[1],'?x',{**env,'subject':'?x'}))
        if text.startswith('not all '):return neg(self.expression(text[4:],env))
        if text.startswith('it is not the case that '):return neg(self.expression(text[25:],env))
        # Consequent-first conditionals.
        for phrase,op in [(' if and only if ','iff'),(' only if ','implies')]:
            if phrase in text:
                a,b=text.split(phrase,1);return (op,self.expression(a,env),self.expression(b,env))
        patterns=[
            r'^if\s+(.+)\s*,?\s+then\s+(.+)$',
            r'^if\s+(.+?)(?:,\s*(?:then\s+)?|\s+then\s+)(.+)$',
            r'^assuming\s+(.+?),\s*(?:we can say(?: that)?|then)\s+(.+)$',
            r'^given that\s+(.+?),\s*(?:it follows that|then)\s+(.+)$',
            r'^based on\s+(.+?),\s*(?:it is evident that|we conclude that)\s+(.+)$',
            r'^when\s+(.+?),\s*(?:we can conclude(?: that)?|then)?\s*(.+)$',
        ]
        for pattern in patterns:
            m=re.match(pattern,text)
            if m:
                local=dict(env);local.setdefault('subject','?x');local['conditional']=True
                # A quantified rule subject remains the antecedent of they/it
                # even when another named subject occurs in a later conjunct.
                if re.search(r'\b(?:someone|something|somebody)\b',m[1]):local['rule_subject']='?x';local['subject']='?x'
                body=self.expression(m[1],local);head=self.expression(m[2],local)
                return ('implies',body,head)
        if text.startswith(('if ','assuming ','given that ','based on ','when ')):
            raise ParseError('Unsupported conditional: '+text)
        # A shared predicate applies separately to coordinated named subjects.
        m=re.match(r'^(.+?)\s+(are|were|both\s+have|both\s+can)\s+(.+)$',text)
        if m and ' and ' in m[1] and not any(w in self.verb_words for w in m[1].split()):
            pred=re.sub(r'^both\s+','',m[2]+' '+m[3])
            return junction('and',*(self.vp(pred,self.term(x,'subject',{}),{}) for x in m[1].split(' and ')))
        m=re.match(r'^(all|every|any|each|no|some)\s+(.+)$',text)
        if m:return self.quantified(m[1],m[2],env)
        m=re.match(r'^none of (?:the )?(.+)$',text)
        if m:return self.quantified('no',m[1],env)
        m=re.match(r'^(everyone|everything|everybody|anyone|anybody|anything|nothing|nobody|someone|something|somebody)\s+(.+)$',text)
        if m and 'subject' not in env:
            q='some' if m[1].startswith('some') else 'no' if m[1] in ('nothing','nobody') else 'all'
            rest=m[2]
            if rest.startswith(('who ','that ')):rest='person '+rest
            elif rest.split()[0] in self.verb_words:rest='person '+rest
            else:rest='person '+rest
            return self.quantified(q,rest,env)
        # Universal and existential copular statements.
        m=re.match(r'^(all|every|any|each|no|some)\s+(.+?)\s+(?:are|is)\s+(.+)$',text)
        if m:
            q,noun,prop=m.groups();v='?x';local={**env,'subject':v}
            restriction=self.noun(noun,v,local);conclusion=self.vp('is '+prop,v,local)
            if q=='no':conclusion=neg(conclusion)
            f=junction('and',restriction,conclusion) if q=='some' else ('implies',restriction,conclusion)
            return ('exists' if q=='some' else 'all',v,f)
        m=re.match(r'^(.+?)\s+(?:things|people|persons|ones)\s+are\s+(.+)$',text)
        if m:
            local={**env,'subject':'?x'}
            return ('all','?x',('implies',self.noun(m[1],'?x',local),self.vp('is '+m[2],'?x',local)))
        m=re.match(r'^(everyone|everything|everybody|someone|something|somebody)\s+(is|are)\s+(.+)$',text)
        if m and 'subject' not in env:
            q='all' if m[1].startswith('every') else 'exists'
            return (q,'?x',self.vp('is '+m[3],'?x',{**env,'subject':'?x'}))
        # Universal/existential rules with a verbal, rather than copular,
        # consequent: "All citizens of X use Y", "Some musicians love music".
        m=re.match(r'^(all|every|any|each|no|some)\s+(.+)$',text)
        if m:
            subject,verbphrase=self.subject_vp(m[2]);local={**env,'subject':'?x'}
            restrict=self.noun(subject,'?x',local);head=self.vp(verbphrase,'?x',local)
            if m[1]=='no':head=neg(head)
            if m[1]=='some':return ('exists','?x',junction('and',restrict,head))
            return ('all','?x',('implies',restrict,head))
        # Fully restated subjects in coordinated clauses.
        for sep,op in [(r',?\s+or\s+','or'),(r',?\s+and\s+','and')]:
            parts=re.split(sep,text)
            if len(parts)>1 and any(re.search(r'\b(?:is|are|was|were)\b',p) for p in parts[1:]):
                first=self.expression(parts[0],env)
                try:s,_=self.subject_vp(parts[0]);subject=self.term(s,'subject',env)
                except ParseError:subject=inherited
                fs=[first]
                for p in parts[1:]:
                    fs.append(self.vp(p,subject,env) if re.match(r'^(?:is|are|was|were|not)\b',p) else self.expression(p,env))
                return junction(op,*fs)
        if inherited is not None:return self.vp(text,inherited,env)
        subj,vp=self.subject_vp(text);s=self.term(subj,'subject',env)
        if self.query_mode and subj.startswith(('a ','an ')) and not env.get('conditional'):
            v='?w';local={**env,'subject':v}
            return ('exists',v,junction('and',self.noun(subj,v,local),self.vp(vp,v,local)))
        if env.get('conditional') and (subj.startswith(('a ','an ')) or subj.split()[0] in ('people','persons','things')):
            noun=re.sub(r'^(?:a|an)\s+','',subj);s=env.get('subject','?x')
            env[noun]=s;env['the '+noun]=s;env['subject']=s
            return junction('and',self.noun(noun,s,env),self.vp(vp,s,env))
        # Bare plural subjects express generic rules; definite noun phrases
        # and known proper names continue to denote individual entities.
        first=subj.split()[0] if subj else ''
        if first not in ('the','a','an') and slug(subj) not in getattr(self,'proper_entities',set()):
            head=re.split(r'\s+(?:who|that|which|in|at|of|with|from)\s+',subj)[0].split()[-1]
            if singular(head)!=head or first in ('people','persons','things'):
                return self.quantified('all',text,env)
        if ' and ' in subj:
            return junction('and',*(self.vp(vp,self.term(x,'subject',env),env) for x in subj.split(' and ')))
        if subj not in self.pronouns and not env.get('rule_subject'):env['subject']=s
        # Pronouns in a rule refer to its subject; separate object variables
        # are assigned in term(), never collapsed into the subject variable.
        if variable(s):env.setdefault('subject',s)
        result=self.vp(vp,s,env)
        if subj.startswith(('a ','an ')):result=junction('and',self.noun(subj,s,env),result)
        return result
    def parse_context(self,context):
        # Newlines are sentence boundaries even when final periods are absent.
        context=re.sub(r'\b(?:facts|rules|premises|knowledge base)\s*:\s*','',context,flags=re.I)
        context=re.sub(r'\b(No|Mr|Mrs|Dr|Prof)\.',r'\1',context)
        context=re.sub(r':\s*(?=if\b)','; ',context,flags=re.I)
        sentences=[s.strip() for s in re.split(r'(?<!\b[A-Z])[.!?]\s+|[\r\n]+',context) if s.strip()]
        # Gather proper multi-word subjects before parsing any sentence. This
        # makes resolution independent of the order of facts and rules.
        for s in sentences:
            m=re.match(r'^([A-Z][\w\'-]*(?:\s+[A-Z][\w\'-]*)*)\s+[a-z]',s)
            if m and m[1] not in ('The','A','An','All','Every','If','No','Some','Any','When','Assuming','Given','Based','People','Everyone','Everything','Someone','Something','Nobody','Nothing','Each','None','Either','Neither') and (not m[1].endswith('s') or m[1].endswith(('es','is','us')) or ' ' in m[1]):
                self.entities.add(slug(m[1]))
        self.proper_entities=set(self.entities)
        formulas=[]
        for sentence in sentences:
            try:
                if ';' in sentence:
                    pieces=sentence.split(';');f=self.expression(pieces[0])
                    subject=self.term(self.subject_vp(pieces[0])[0])
                    f=junction('and',f,*(self.expression(piece,{'subject':subject}) for piece in pieces[1:]))
                else:f=self.expression(sentence)
                # Rule variables not explicitly quantified are universally scoped.
                free=formula_free_vars(f)
                for v in sorted(free,reverse=True):f=('all',v,f)
                formulas.append(f)
            except (ParseError,IndexError) as e:self.diagnostics.append(str(e))
        # Intersective noun phrases entail their independently mentioned
        # modifiers/heads ("famous athlete" -> "famous" and "athlete").
        # Content clauses, relational phrases and non-intersective modifiers
        # are excluded. These rules use only vocabulary in this context.
        formulas.extend(self.lexical_axioms())
        return formulas
    def lexical_axioms(self):
        formulas=[]
        for phrase in sorted(self.nominals):
            words=phrase.split('_')
            if len(words)<2 or any(w in ('of','in','at','to','for','from','on','with','that','who','which','former','fake','alleged','possible','would','wouldbe','toy') for w in words):continue
            for n in range(1,len(words)):
                for part in ('_'.join(words[:n]),'_'.join(words[n:])):
                    if part in self.nominals:
                        formulas.append(('all','?type',('implies',atom(phrase,'?type'),atom(part,'?type'))))
        return formulas

def formula_free_vars(f):
    if f[0]=='atom':return set().union(*(term_vars(t) for t in f[2])) if f[2] else set()
    if f[0] in ('all','exists'):return formula_free_vars(f[2])-{f[1]}
    return set().union(*(formula_free_vars(x) for x in f[1:])) if f[1:] else set()




## Indexed forward chaining and proof provenance

In [5]:
def signed_literal(f):
    if f[0]=='atom':return True,f
    if f[0]=='not' and f[1][0]=='atom':return False,f[1]
    raise ParseError('Not a literal')

def horn_program(formulas):
    facts=[];rules=[]
    def add(f):
        if f[0]=='all':return add(f[2])
        if f[0]=='and':
            for x in f[1:]:add(x)
            return
        if f[0]=='implies':
            body=list(f[1][1:]) if f[1][0]=='and' else [f[1]]
            body=[signed_literal(b) for b in body]
            heads=f[2][1:] if f[2][0]=='and' else [f[2]]
            for h in heads:rules.append((body,signed_literal(h),show(f)))
            return
        lit=signed_literal(f)
        if formula_free_vars(f):rules.append(([],lit,show(f)))
        else:facts.append(lit)
    for f in formulas:add(f)
    return facts,rules

class HornModel:
    """Complete finite function-free Horn closure; explicit signed facts.

    Optional CWA applies stratified negation-as-failure, explicitly requested
    for datasets with that semantics. OWA is the general entailment default.
    """
    def __init__(self,formulas,entities,mode='owa',timeout=1.0):
        self.mode=mode;self.model=set();self.index=defaultdict(set);self.provenance={}
        self.rounds=0;self.deadline=time.perf_counter()+timeout
        facts,self.rules=horn_program(formulas)
        self.entities=set(entities)
        for sign,a in facts:
            self.entities.update(t for t in a[2] if not variable(t))
            self.insert((sign,a),('fact',[]))
        for body,(_,head),_ in self.rules:
            for a in [head]+[a for _,a in body]:
                self.entities.update(t for t in a[2] if not variable(t))
        self.entities=self.entities or {'$domain0'}
        levels=defaultdict(int)
        if mode=='cwa':
            # Explicit negative heads have their own stratum key. They do not
            # derive the positive predicate tested by negation-as-failure.
            predicates={(True,a[1]) for b,_,_ in self.rules for _,a in b}|{(s,h[1]) for _,(s,h),_ in self.rules}
            for k in range(len(predicates)+1):
                changed=False
                for body,(headsign,h),_ in self.rules:
                    need=max([levels[(True,a[1])]+(not sign) for sign,a in body]+[0])
                    if need>levels[(headsign,h[1])]:levels[(headsign,h[1])]=need;changed=True
                if not changed:break
            else:
                # Predicate cycles can disappear after grounding or after
                # evaluating facts. Compute the well-founded partial model;
                # queries that remain undefined are explicitly rejected.
                self.well_founded_closure();return
        for level in sorted({levels[(s,h[1])] for _,(s,h),_ in self.rules}):
            active=[r for r in self.rules if levels[(r[1][0],r[1][1][1])]==level]
            changed=True
            while changed:
                changed=False;self.rounds+=1
                if time.perf_counter()>self.deadline:raise ResourceLimit('Horn closure time limit')
                for body,(sign,head),label in active:
                    positive=[b for b in body if b[0] or mode=='owa']
                    negative=[b for b in body if not b[0] and mode=='cwa']
                    substitutions=[({},[])]
                    for s,a in positive:
                        next_sub=[]
                        for sub,used in substitutions:
                            target=formula_sub(a,sub)
                            for fact in sorted(self.index[(s,a[1],len(a[2]))],key=repr):
                                merged=unify_atoms(target,fact,sub)
                                if merged is not None:next_sub.append((merged,used+[(s,fact)]))
                        substitutions=next_sub
                    allvars=formula_free_vars(head)|set().union(*(formula_free_vars(a) for _,a in negative))
                    for sub,used in substitutions:
                        free=sorted(v for v in allvars if variable(term_sub(v,sub)))
                        for values in itertools.product(sorted(self.entities,key=repr),repeat=len(free)):
                            full={**sub,**dict(zip(free,values))}
                            if any((True,formula_sub(a,full)) in self.model for _,a in negative):continue
                            derived=(sign,formula_sub(head,full))
                            if formula_free_vars(derived[1]):continue
                            used_neg=[('NAF',formula_sub(a,full)) for _,a in negative]
                            changed=self.insert(derived,(label,used+used_neg)) or changed
    def well_founded_closure(self):
        """Finite alternating fixpoint; Gamma uses an indexed positive agenda.

        Reference: Van Gelder, JCSS 47 (1993), alternating fixpoint semantics.
        Lower atoms are true, atoms outside upper are false, the rest unknown.
        Explicit negative heads remain separate signed atoms, as in CWA above.
        """
        initial=set(self.model);ground=[];universe=set(initial)
        for body,head,label in self.rules:
            variables=sorted(set().union(*(formula_free_vars(a) for _,a in body+[head])))
            if any(isinstance(t,tuple) for _,a in body+[head] for t in a[2]):raise ParseError('CWA requires function-free terms')
            for values in itertools.product(sorted(self.entities,key=repr),repeat=len(variables)):
                if len(ground)>=100000 or time.perf_counter()>self.deadline:raise ResourceLimit('CWA grounding limit')
                sub=dict(zip(variables,values));h=(head[0],formula_sub(head[1],sub))
                positive=frozenset((True,formula_sub(a,sub)) for sign,a in body if sign)
                negative=frozenset((True,formula_sub(a,sub)) for sign,a in body if not sign)
                ground.append((positive,negative,h,label));universe.update(positive|negative|{h})
        def gamma(assumed):
            found=set(initial);why={lit:('fact',[]) for lit in initial};watch=defaultdict(list);remaining={};agenda=list(initial)
            for i,(positive,negative,head,label) in enumerate(ground):
                if negative & assumed:continue
                remaining[i]=len(positive)
                for lit in positive:watch[lit].append(i)
                if not positive and head not in found:
                    found.add(head);agenda.append(head);why[head]=(label,[('NAF',a) for _,a in negative])
            cursor=0
            while cursor<len(agenda):
                if time.perf_counter()>self.deadline:raise ResourceLimit('Well-founded closure time limit')
                lit=agenda[cursor];cursor+=1
                for i in watch[lit]:
                    remaining[i]-=1
                    if remaining[i]==0:
                        positive,negative,head,label=ground[i]
                        if head not in found:
                            found.add(head);agenda.append(head);why[head]=(label,list(positive)+[('NAF',a) for _,a in negative])
            return found,why
        lower=set();upper=universe
        while True:
            self.rounds+=1;new_lower,why=gamma(upper);new_upper,_=gamma(new_lower)
            if new_lower==lower and new_upper==upper:break
            lower,upper=new_lower,new_upper
        self.undefined=upper-lower;self.well_founded=True
        for lit in lower:self.insert(lit,why[lit])
    def insert(self,lit,provenance):
        if lit in self.model:return False
        self.model.add(lit);sign,a=lit;self.index[(sign,a[1],len(a[2]))].add(a)
        self.provenance[lit]=provenance;return True
    def holds(self,f):
        if f[0]=='and':return all(self.holds(x) for x in f[1:])
        if f[0]=='or':return any(self.holds(x) for x in f[1:])
        sign,a=signed_literal(f)
        if self.mode=='cwa' and (True,a) in getattr(self,'undefined',set()):
            raise ParseError('Negative dependency cycle leaves query undefined under well-founded semantics')
        if not sign and self.mode=='cwa':return (True,a) not in self.model
        return (sign,a) in self.model
    def explain(self,f,depth=0,seen=None):
        seen=set() if seen is None else seen
        try:lit=signed_literal(f)
        except ParseError:return [show(f)+' = '+str(self.holds(f))]
        if lit in seen:return ['  '*depth+show(f)+' [already shown]']
        seen.add(lit)
        if lit not in self.provenance:
            if self.mode=='cwa' and not lit[0] and (True,lit[1]) not in self.model:
                return ['  '*depth+show(f)+' [positive atom false in completed CWA model]']
            return ['  '*depth+show(f)+' [not derived]']
        label,used=self.provenance[lit]
        out=['  '*depth+show(f)+' <= '+label]
        for sign,a in used:
            if sign=='NAF':out.append('  '*(depth+1)+'not '+show(a)+' [false in completed CWA model]')
            else:out.extend(self.explain(a if sign else neg(a),depth+1,seen))
        return out




## Structural causal models and counterfactual inference

In [6]:
class BooleanParser:
    def parse(self,text):
        text=re.sub(r'\btogether\b','',text.lower())
        self.t=re.findall(r'\w+|[()]',text);self.i=0
        if not self.t:raise ParseError('Empty Boolean expression')
        v=self.disjunction()
        if self.i!=len(self.t):raise ParseError('Unparsed causal expression: '+text)
        return v
    def peek(self):return self.t[self.i] if self.i<len(self.t) else ''
    def disjunction(self):
        out=[self.conjunction()]
        while self.peek()=='or':self.i+=1;out.append(self.conjunction())
        return junction('or',*out)
    def conjunction(self):
        out=[self.unary()]
        while self.peek()=='and':self.i+=1;out.append(self.unary())
        return junction('and',*out)
    def unary(self):
        t=self.peek();self.i+=1
        if t=='not':return neg(self.unary())
        if t=='(':
            v=self.disjunction()
            if self.peek()!=')':raise ParseError('Unclosed Boolean expression')
            self.i+=1;return v
        if not t or t in ('and','or',')'):raise ParseError('Expected Boolean variable')
        return atom(t)

def boolean_vars(f):
    if f[0]=='atom':return {f[1]}
    return set().union(*(boolean_vars(x) for x in f[1:]))

def boolean_value(f,state):
    if f[0]=='atom':return state[f[1]]
    if f[0]=='not':return not boolean_value(f[1],state)
    if f[0]=='and':return all(boolean_value(x,state) for x in f[1:])
    if f[0]=='or':return any(boolean_value(x,state) for x in f[1:])
    raise ParseError('Unknown causal operator')

class CausalModel:
    def __init__(self,text):
        self.equations={};self.priors={};self.observed={};self.trace=[]
        self.abducted=None
        text=text.strip()
        for m in re.finditer(r'([A-Za-z]\w*)\s*~\s*Bern\s*\(\s*([0-9.]+)\s*\)',text,re.I):
            p=float(m[2])
            if not 0<=p<=1:raise ParseError('Invalid Bernoulli probability')
            self.priors[m[1].lower()]=p
        for m in re.finditer(r'We observed\s+([^.]*)',text,re.I):
            obs=re.sub(r'\b(\w+)\s+(?:does\s+)?not\s+occurs?\b',r'not \1',m[1],flags=re.I)
            obs=re.sub(r'\b(?:occurred|occurs?|happened|and)\b',' ',obs,flags=re.I)
            for v in re.finditer(r'\b(not\s+)?([A-Za-z]\w*)\b',obs,re.I):
                self.observed[v[2].lower()]=not bool(v[1])
        text=re.sub(r'We observed\s+[^.]*\.?','',text,flags=re.I)
        text=re.sub(r'\w+\s*~\s*Bern\s*\([0-9.]+\)\.?','',text,flags=re.I)
        text=re.sub(r'We know that\s+','',text,flags=re.I).strip(' .')
        # Split before a new causal clause, retaining comma-separated children.
        pattern=r'[,.;]\s*(?:and\s+)?(?=(?:not\s+)?\w+(?:\s+(?:and|or)\s+(?:not\s+)?\w+)*(?:\s+together)?\s+causes?\b)'
        clauses=re.split(pattern,text,flags=re.I)
        for c in clauses:
            c=c.strip(' .;,')
            if not c:continue
            m=re.fullmatch(r'(.*?)\s+causes?\s+(.+)',c,re.I)
            if not m:raise ParseError('Unsupported causal clause: '+c)
            parent=BooleanParser().parse(m[1])
            children=re.split(r',\s*(?:and\s+)?|\s+and\s+',m[2])
            for child in children:
                child=child.strip(' .')
                if not child:continue
                negative=child.lower().startswith('not ')
                name=child[4:].strip().lower() if negative else child.lower()
                if not re.fullmatch(r'[a-z]\w*',name):raise ParseError('Invalid causal child '+name)
                expr=neg(parent) if negative else parent
                if name in self.equations and self.equations[name]!=expr:raise ParseError('Multiple incompatible equations for '+name)
                self.equations[name]=expr
        if not self.equations:raise ParseError('No causal equations')
        variables=set(self.equations)|set(self.priors)|set(self.observed)
        for f in self.equations.values():variables.update(boolean_vars(f))
        self.roots=sorted(variables-set(self.equations))
        if len(self.roots)>18:raise ResourceLimit('Too many exogenous Boolean variables')
        # Explicitly reject cycles; evaluation never guesses a fixed point.
        remaining=set(self.equations);known=set(self.roots);self.order=[]
        while remaining:
            ready=sorted(v for v in remaining if boolean_vars(self.equations[v])<=known)
            if not ready:raise ParseError('Cyclic causal model')
            self.order.extend(ready);known.update(ready);remaining.difference_update(ready)
    def world(self,roots,interventions):
        state={**roots,**interventions}
        for v in self.order:
            if v not in interventions:state[v]=boolean_value(self.equations[v],state)
        return state
    def predict(self,question):
        q=clean_nl(question)
        m=re.search(r'would\s+(not\s+)?(\w+)\s+occur',q)
        if not m:raise ParseError('Unsupported counterfactual query')
        target=m[2];target_sign=not bool(m[1]);interventions={}
        # Exclude the replaced factual alternative from the action clause.
        action=re.sub(r'\binstead of\s+(?:not\s+)?\w+','',q)
        for match in re.finditer(r'\b(?:if|assume|suppose|and)\s+(not\s+)?([a-z]\w*)',action):
            v=match[2]
            if v in self.equations or v in self.roots:interventions[v]=not bool(match[1])
        if not interventions:raise ParseError('No supported intervention found')
        if self.abducted is None:
            self.abducted=[]
            for values in itertools.product((False,True),repeat=len(self.roots)):
                roots=dict(zip(self.roots,values));weight=1.0
                for v,value in roots.items():
                    p=self.priors.get(v,0.5);weight*=p if value else 1-p
                if not weight:continue
                factual=self.world(roots,{})
                if any(factual.get(v)!=value for v,value in self.observed.items()):continue
                self.abducted.append((roots,weight))
        mass=0.0;yes=0.0;worlds=0
        for roots,weight in self.abducted:
            cf=self.world(roots,interventions)
            if target not in cf:raise ParseError('Unknown causal target '+target)
            worlds+=1;mass+=weight
            if cf[target]==target_sign:yes+=weight
        if mass==0:raise ParseError('Factual observations are inconsistent with the causal equations')
        probability=yes/mass
        trace=[f'Abduction: {worlds} exogenous assignments consistent with observations {self.observed}.',
               f'Action: do({interventions}); replace intervened equations.',
               'Deduction: recompute endogenous variables in topological order.',
               f'P({target}={target_sign} | observations, intervention) = {probability:.6f}.']
        if any(v not in self.priors for v in self.roots):trace.append('Unspecified exogenous priors use independent Bernoulli(0.5); deterministic outcomes do not depend on this choice.')
        trace.append('Binary causal decision uses probability > 0.5; exact probability is returned separately.')
        return probability>0.5,probability,trace




## Unified API and input/output

In [7]:
@dataclass
class Result:
    prediction: bool
    method: str
    status: str
    parse_seconds: float=0.0
    inference_seconds: float=0.0
    total_seconds: float=0.0
    trace: list=field(default_factory=list)
    diagnostics: list=field(default_factory=list)
    probability: Optional[float]=None
    stats: dict=field(default_factory=dict)

class ReasoningAgent:
    def __init__(self,mode='owa',timeout=1.0,cache_size=256):
        if mode not in ('owa','cwa'):raise ValueError('mode must be owa or cwa')
        self.mode=mode;self.timeout=timeout;self.cache_size=cache_size;self.cache=OrderedDict()
        self.parse_cache=OrderedDict();self.causal_cache=OrderedDict()
    def clear_cache(self):self.cache.clear();self.parse_cache.clear();self.causal_cache.clear()
    def predict(self,context,statement,*,premises_fol=None,conclusion_fol=None,mode=None):
        start=time.perf_counter();parse_time=0.0;method='nl';diagnostics=[]
        mode=mode or self.mode
        try:
            if re.search(r'\bcauses?\b',context,re.I) and re.search(r'\bwould\b.*\boccur\b',statement,re.I):
                cached=context in self.causal_cache
                if cached:model=self.causal_cache.pop(context);self.causal_cache[context]=model
                else:
                    model=CausalModel(context)
                    if self.cache_size>0:
                        self.causal_cache[context]=model
                        while len(self.causal_cache)>self.cache_size:self.causal_cache.popitem(last=False)
                parse_time=time.perf_counter()-start
                pred,prob,trace=model.predict(statement)
                total=time.perf_counter()-start
                return Result(pred,'causal_enumeration','entailed' if prob==1 else 'not_entailed' if prob==0 else 'probabilistic',parse_time,total-parse_time,total,trace,[],prob,{'cache_hit':cached})
            if premises_fol is not None and conclusion_fol is not None:
                method='fol_ground_refutation'
                lines=premises_fol if isinstance(premises_fol,list) else premises_fol.splitlines()
                premises=[]
                for line in lines:
                    if not line.strip():continue
                    try:premises.append(FormulaParser().parse(line))
                    except ParseError as e:diagnostics.append('Whole premise omitted: '+str(e))
                query=FormulaParser().parse(conclusion_fol)
                parse_time=time.perf_counter()-start
                pred,trace,stats=prove_fol(premises,query,timeout=self.timeout)
                total=time.perf_counter()-start
                status=('entailed_partial_input' if diagnostics else 'entailed') if pred else ('unsupported_input' if diagnostics else 'not_proved')
                return Result(pred,method,status,parse_time,total-parse_time,total,trace,diagnostics,None,stats)
            parser=NLParser()
            if context in self.parse_cache:
                cached_parse=self.parse_cache.pop(context);self.parse_cache[context]=cached_parse
                premises,state=cached_parse;premises=list(premises)
                for name,value in state.items():
                    setattr(parser,name,set(value) if isinstance(value,frozenset) else list(value) if isinstance(value,tuple) else value)
            else:
                premises=parser.parse_context(context)
                if self.cache_size>0:
                    state={name:frozenset(getattr(parser,name)) for name in ('entities','proper_entities','nominals')}
                    state.update(classical=parser.classical,diagnostics=tuple(parser.diagnostics))
                    self.parse_cache[context]=(tuple(premises),state)
                    while len(self.parse_cache)>self.cache_size:self.parse_cache.popitem(last=False)
            parser.query_mode=True;query=parser.expression(statement)
            for v in sorted(formula_free_vars(query),reverse=True):query=('all',v,query)
            premises=list(dict.fromkeys(premises+parser.lexical_axioms()))
            diagnostics=parser.diagnostics
            parse_time=time.perf_counter()-start
            # If some input text is malformed, positive support from the
            # successfully parsed subset is still sound. Do not use absence
            # as evidence under CWA until every premise was parsed.
            closure_mode='owa' if mode=='cwa' and diagnostics else mode
            # A proof from a correctly parsed subset remains valid. Unparsed
            # whole premises are diagnosed; individual conditions stay intact.
            key=(tuple(premises),tuple(sorted(parser.entities)),closure_mode)
            try:
                if key in self.cache:
                    model=self.cache.pop(key);self.cache[key]=model;cached=True
                else:
                    model=HornModel(premises,parser.entities,closure_mode,self.timeout);cached=False
                    if self.cache_size>0:
                        self.cache[key]=model
                        while len(self.cache)>self.cache_size:self.cache.popitem(last=False)
                pred=model.holds(query);trace=model.explain(query)
                stats={'facts':len(model.model),'rules':len(model.rules),'rounds':model.rounds,'cache_hit':cached}
                method='horn_'+mode
                if not pred and parser.classical and mode=='owa':
                    # Categorical and coordinated English can require classical
                    # contraposition or a proof by cases beyond Horn closure.
                    pred,trace,stats=prove_fol(premises,query,timeout=self.timeout)
                    method='nl_fol_ground_refutation'
            except ParseError as error:
                if mode=='cwa':raise
                method='nl_fol_ground_refutation'
                pred,trace,stats=prove_fol(premises,query,timeout=self.timeout)
            total=time.perf_counter()-start
            status=('entailed_partial_input' if diagnostics else 'entailed') if pred else ('unsupported_input' if diagnostics else 'not_proved')
            return Result(pred,method,status,parse_time,total-parse_time,total,trace,diagnostics,None,stats)
        except (ParseError,ResourceLimit,RecursionError) as error:
            total=time.perf_counter()-start
            status='resource_limit' if isinstance(error,(ResourceLimit,RecursionError)) else 'unsupported_input'
            return Result(False,method,status,parse_time,max(0,total-parse_time),total,[],diagnostics+[str(error)])
    def predict_record(self,row,formal_annotations=False,mode=None):
        context=row.get('context',row.get('premises',row.get('given_info',row.get('theory',''))))
        if isinstance(context,list):context='\n'.join(context)
        query=row.get('statement',row.get('question',row.get('conclusion',row.get('query',''))))
        extra={}
        if formal_annotations and row.get('premises-FOL') and row.get('conclusion-FOL'):
            extra={'premises_fol':row['premises-FOL'],'conclusion_fol':row['conclusion-FOL']}
        return self.predict(str(context),str(query),mode=mode,**extra)

def binary_label(value):
    if isinstance(value,bool):return value
    if isinstance(value,(int,float)) and value in (0,1):return bool(value)
    value=str(value).strip().lower()
    if value in ('true','yes','1','entailment','entailed'):return True
    if value in ('false','no','0','not entailment','not entailed','contradiction','uncertain','unknown'):return False
    raise ValueError('Unsupported label: '+value)

def read_records(path):
    path=Path(path)
    with path.open(encoding='utf-8-sig',newline='') as stream:
        if path.suffix.lower()=='.csv':return list(csv.DictReader(stream))
        if path.suffix.lower()=='.jsonl':return [json.loads(line) for line in stream if line.strip()]
        data=json.load(stream)
        if isinstance(data,dict) and 'rows' in data:return [x.get('row',x) for x in data['rows']]
        if isinstance(data,list):return data
        raise ValueError('Expected CSV, JSON list, JSONL, or HF rows response')

def write_submission(records,path,agent=None,formal_annotations=False,diagnostics_path=None):
    agent=agent or ReasoningAgent();output=[];records=list(records)
    batch=getattr(agent,'predict_records',None)
    results=batch(records,formal_annotations=formal_annotations) if batch else (agent.predict_record(row,formal_annotations=formal_annotations) for row in records)
    diagnostics=[]
    for i,(row,result) in enumerate(zip(records,results)):
        identifier=row.get('id',row.get('example_id',row.get('question_id',i)))
        output.append({'id':identifier,'predicted_label':str(result.prediction)})
        if diagnostics_path:diagnostics.append({'id':identifier,**asdict(result)})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',newline='',encoding='utf-8') as stream:
        writer=csv.DictWriter(stream,fieldnames=['id','predicted_label']);writer.writeheader();writer.writerows(output)
    if diagnostics_path:Path(diagnostics_path).write_text(json.dumps(diagnostics,indent=2,ensure_ascii=False),encoding='utf-8')
    return output


## Optional general resolution search

In [8]:
"""Bounded first-order resolution with factoring and age/weight clause search.

This is an independently implemented classical reasoner. It can prove entailment
without finite-domain grounding. A stopped search never asserts non-entailment.
"""
import heapq
import itertools
import time
from collections import defaultdict, deque

def canonical_clause(clause):
    clause=frozenset(clause)
    if any((not sign,a) in clause for sign,a in clause):return None
    rename={}
    for _,a in sorted(clause,key=lambda x:(x[1][1],len(x[1][2]),x[0],repr(x[1][2]))):
        for v in sorted(formula_free_vars(a)):
            if v not in rename:rename[v]='?v'+str(len(rename))
    return frozenset((sign,formula_sub(a,rename)) for sign,a in clause)

def equality_clauses(clauses):
    atoms=[a for c in clauses for _,a in c]
    if not any(a[1]=='=' for a in atoms):return []
    x,y,z='?eqx','?eqy','?eqz'
    out=[frozenset([(True,atom('=',x,x))]),
         frozenset([(False,atom('=',x,y)),(True,atom('=',y,x))]),
         frozenset([(False,atom('=',x,y)),(False,atom('=',y,z)),(True,atom('=',x,z))])]
    predicates={(a[1],len(a[2])) for a in atoms if a[1]!='='};functions=set()
    def collect(t):
        if isinstance(t,tuple):
            functions.add((t[1],len(t[2])))
            for a in t[2]:collect(a)
    for a in atoms:
        for t in a[2]:collect(t)
    for name,arity in sorted(predicates|functions):
        left=tuple(f'?a{i}' for i in range(arity));right=tuple(f'?b{i}' for i in range(arity))
        prefix=[(False,atom('=',a,b)) for a,b in zip(left,right)]
        if (name,arity) in predicates:out.append(frozenset(prefix+[(False,atom(name,*left)),(True,atom(name,*right))]))
        if (name,arity) in functions:out.append(frozenset(prefix+[(True,atom('=',('fn',name,left),('fn',name,right)))]))
    return out

def term_depth(t):return 1+max((term_depth(x) for x in t[2]),default=0) if isinstance(t,tuple) else 0

def prove_resolution(premises,query,timeout=1.0,max_clauses=3000,max_width=16,max_term_depth=5):
    start=time.perf_counter();deadline=start+timeout
    clauses,_=clausify(list(premises)+[neg(query)])
    clauses+=equality_clauses(clauses)
    pool=[];parents=[];seen={};active=set();pending=set();index=defaultdict(set);heap=[];ages=deque()
    stats={'selected':0,'resolutions':0,'factors':0,'generated':0,'subsumed':0,'pruned':0}
    query_predicates=set()
    def query_atoms(f):
        if f[0]=='atom':query_predicates.add(f[1])
        elif f[0] in ('all','exists'):query_atoms(f[2])
        else:
            for x in f[1:]:query_atoms(x)
    query_atoms(query)
    def add(c,ancestry):
        c=canonical_clause(c)
        if c is None or c in seen:return None
        if len(c)>max_width or any(term_depth(t)>max_term_depth for _,a in c for t in a[2]):stats['pruned']+=1;return None
        # Exact literal subset subsumption is sound even with variables.
        candidates=set()
        for lit in c:candidates.update(index.get((lit[0],lit[1][1],len(lit[1][2])),set()))
        if any(pool[i]<=c for i in candidates if len(pool[i])<=len(c)):
            stats['subsumed']+=1;return None
        if len(pool)>=max_clauses:raise ResourceLimit('Resolution clause budget')
        i=len(pool);pool.append(c);parents.append(ancestry);seen[c]=i;pending.add(i);ages.append(i)
        weight=4*len(c)+sum(term_depth(t) for _,a in c for t in a[2])
        if any(a[1] in query_predicates for _,a in c):weight-=1
        heapq.heappush(heap,(weight,i));stats['generated']+=1
        return i
    def proof(i):
        needed=set()
        def visit(j):
            if j in needed:return
            needed.add(j)
            for ancestor in parents[j][1]:visit(ancestor)
        visit(i);trace=[]
        for j in sorted(needed):
            rendered=' OR '.join(('' if sign else 'NOT ')+show(a) for sign,a in sorted(pool[j],key=repr)) or 'EMPTY'
            trace.append(f'C{j}: {rendered} [{parents[j][0]}; parents={parents[j][1]}]')
        return trace
    def renamed(c,tag):
        variables=sorted(set().union(*(formula_free_vars(a) for _,a in c)))
        sub={v:f'?apart{tag}_{j}' for j,v in enumerate(variables)}
        return frozenset((s,formula_sub(a,sub)) for s,a in c)
    def resolve(c,d):
        for s,a in c:
            for t,b in d:
                if s==t:continue
                sub=unify_atoms(a,b)
                if sub is None:continue
                stats['resolutions']+=1
                yield frozenset((p,formula_sub(f,sub)) for p,f in (c-{(s,a)})|(d-{(t,b)}))
    try:
        for c in clauses:
            i=add(c,('input',[]))
            if i is not None and not pool[i]:return True,proof(i),dict(stats,status='entailed')
        while pending:
            if time.perf_counter()>deadline:raise ResourceLimit('Resolution time budget')
            if stats['selected']%6==5:
                while ages and ages[0] not in pending:ages.popleft()
                i=ages.popleft()
            else:
                while heap and heap[0][1] not in pending:heapq.heappop(heap)
                _,i=heapq.heappop(heap)
            pending.remove(i);c=pool[i];stats['selected']+=1
            # Factoring is needed for first-order completeness.
            for (s,a),(t,b) in itertools.combinations(c,2):
                if s!=t:continue
                sub=unify_atoms(a,b)
                if sub is not None:
                    factor=frozenset((p,formula_sub(f,sub)) for p,f in c-{(t,b)})
                    j=add(factor,('factor',[i]));stats['factors']+=1
                    if j is not None and not pool[j]:return True,proof(j),dict(stats,status='entailed')
            partners=set()
            for s,a in c:partners.update(index[(not s,a[1],len(a[2]))])
            for partner in sorted(partners):
                if time.perf_counter()>deadline:raise ResourceLimit('Resolution time budget')
                d=renamed(pool[partner],f'{i}_{partner}')
                for resolvent in resolve(c,d):
                    j=add(resolvent,('resolve',[i,partner]))
                    if j is not None and not pool[j]:return True,proof(j),dict(stats,status='entailed')
            active.add(i)
            for s,a in c:index[(s,a[1],len(a[2]))].add(i)
        return False,['Resolution search saturated without an empty clause.'],dict(stats,status='not_proved' if not stats['pruned'] else 'resource_limit')
    except ResourceLimit as error:
        return False,[str(error)],dict(stats,status='resource_limit')


## Validated local language-to-logic parser

In [9]:
"""Optional local English-to-FOL translator. It never returns a truth label.

Only context and query strings are sent to a loopback llama.cpp server. Model
output is schema checked, parsed, and passed to the independent classical engine.
"""
import hashlib
import json
import re
import time
import urllib.request
from urllib.parse import urlparse

PROMPT_VERSION='fol-translation-v4'
SYSTEM_PROMPT='''You translate English statements into first-order logic. You are a semantic parser, not a question-answering system. Never decide entailment, output True/False labels, prove anything, or add derived facts. Translate only what each input sentence says. Do not use outside knowledge. Preserve negation, conjunction, disjunction, restrictions, and quantifier scope.

Return ONLY the requested JSON. Each source premise has an index; return exactly one formula for each index, combining its clauses if necessary. Each query also has an index; translate it without answering it. Use the SAME predicate names, argument order, and constant spelling everywhere. Canonicalize paraphrases that have the same meaning, not different concepts.

Syntax: P(alice), R(alice,bob), ¬P(alice), (P(alice) ∧ Q(alice)), (P(alice) ∨ Q(alice)), (P(alice) → Q(alice)), (P(alice) ↔ Q(alice)), (P(alice) ⊕ Q(alice)), ∀x (P(x) → Q(x)), ∃x (P(x) ∧ Q(x)), x=y. No prose, no backticks, no labels, no function names for logical connectives. Fully parenthesize every binary formula and quantify all variable occurrences. Use ASCII snake_case predicate names and constants; identifiers may contain digits. Avoid arithmetic: represent comparisons as predicates.

Translation rules:
- All A are B: ∀x (A(x) → B(x)). No A are B: ∀x (A(x) → ¬B(x)).
- Some A are B: ∃x (A(x) ∧ B(x)). Do NOT assume that a universal statement implies existence.
- Someone has P and someone has Q: two separate existential quantifiers unless the text identifies the same person.
- Either A or B is inclusive ∨ unless the text says not both/exactly one. Neither A nor B means (¬A ∧ ¬B).
- A only if B means A → B. A if B means B → A. Not all A are B means ¬∀x (A(x) → B(x)).
- Restricted subjects keep every restriction: All artists who sing are happy means ∀x ((Artist(x) ∧ Sing(x)) → Happy(x)).
- Relative clauses add properties of their referent. Share the referent across pronouns. Keep different named people distinct syntactically; do not assert inequality unless stated.
- Do not translate an English conditional query by trying to prove it. Preserve the entire antecedent and consequent.
- Do not add an answer as an extra premise. Do not copy a query into premises unless it is actually stated in the input.
- Preserve EVERY fact in a sentence. "The tower is a tall building in Harbor City, built in 2001" has four facts: building(tower), tall(tower), in(tower,harbor_city), built_in(tower,2001). Do not omit its location or date.
- Do not collapse entities. "Mira owns a bicycle" is ∃x (bicycle(x) ∧ owns(mira,x)), NOT bicycle(mira). "Mira sings a song" is ∃x (song(x) ∧ sings(mira,x)), NOT song(mira) or sings(mira). Referents such as projects, songs and buildings remain separate from their owners or users.
- "If a song is not slow, Mira does not sing it" is ∀x ((song(x) ∧ ¬slow(x)) → ¬sings(mira,x)). Never replace the variable with a universal constant named song.
- A query about "a song Mira sings" existentially quantifies the song. A query about every song universally quantifies it. Do not treat these as the same query.
- Every property of a coordinated/relative description must appear. "A quiet musician who lives in a city" retains quiet, musician, lives_in and city.

Example input premises: ["All pilots are travelers.", "Nora is a pilot or a sailor.", "Nora is not a sailor."]
Queries: ["Nora is a traveler.", "Every traveler is a pilot."]
Example output: {"premises":[{"index":0,"formula":"∀x (pilot(x) → traveler(x))"},{"index":1,"formula":"(pilot(nora) ∨ sailor(nora))"},{"index":2,"formula":"¬sailor(nora)"}],"queries":[{"index":0,"formula":"traveler(nora)"},{"index":1,"formula":"∀x (traveler(x) → pilot(x))"}]}

Example input premises: ["A researcher uses a device.", "Every device that a researcher uses is calibrated.", "Iona is a researcher."]
Queries: ["Iona uses a device.", "Some device is calibrated."]
Example output: {"premises":[{"index":0,"formula":"∃x ∃y ((researcher(x) ∧ device(y)) ∧ uses(x,y))"},{"index":1,"formula":"∀x ∀y (((device(y) ∧ researcher(x)) ∧ uses(x,y)) → calibrated(y))"},{"index":2,"formula":"researcher(iona)"}],"queries":[{"index":0,"formula":"∃x (device(x) ∧ uses(iona,x))"},{"index":1,"formula":"∃x (device(x) ∧ calibrated(x))"}]}
'''

ITEM={'type':'object','properties':{'index':{'type':'integer','minimum':0},'formula':{'type':'string'}},'required':['index','formula'],'additionalProperties':False}
SCHEMA={'type':'object','properties':{'premises':{'type':'array','items':ITEM},'queries':{'type':'array','items':ITEM}},'required':['premises','queries'],'additionalProperties':False}

def split_premises(context):
    if isinstance(context,list):return [str(s).strip() for s in context if str(s).strip()]
    context=re.sub(r'\b(?:facts|rules|premises|knowledge base)\s*:\s*','',str(context),flags=re.I)
    # Existing line boundaries take precedence, avoiding initials/abbreviations.
    lines=[line.strip() for line in context.splitlines() if line.strip()]
    if len(lines)>1:return lines
    return [part.strip() for part in re.split(r'(?<=[.!?])\s+(?=[A-Z"“])',context) if part.strip()]

def validate_translation(data,premises,queries):
    if not isinstance(data,dict) or set(data)!= {'premises','queries'}:raise ParseError('Invalid translator output fields')
    parsed={};arities={};errors=[]
    def visit_term(term):
        if isinstance(term,tuple):
            for argument in term[2]:visit_term(argument)
        elif term in ('x','y','z','u','v','w'):
            raise ParseError('Unbound variable-like constant: '+term)
    def visit(f):
        if f[0]=='atom':
            key=f[1];arity=len(f[2])
            if key in arities and arities[key]!=arity:raise ParseError('Inconsistent predicate arity: '+key)
            arities[key]=arity
            for term in f[2]:visit_term(term)
        elif f[0] in ('all','exists'):visit(f[2])
        else:
            for part in f[1:]:visit(part)
    for kind,source in (('premises',premises),('queries',queries)):
        entries=data[kind]
        if not isinstance(entries,list) or len(entries)!=len(source):raise ParseError('Incomplete translation coverage: '+kind)
        if any(not isinstance(x,dict) or set(x)!={'index','formula'} for x in entries):raise ParseError('Invalid translation item')
        if sorted(x['index'] for x in entries)!=list(range(len(source))):raise ParseError('Missing or duplicate source index')
        parsed[kind]=[]
        for item in sorted(entries,key=lambda x:x['index']):
            try:
                if not isinstance(item['formula'],str) or len(item['formula'])>12000:raise ParseError('Invalid formula text')
                f=FormulaParser().parse(item['formula'])
                if formula_free_vars(f):raise ParseError('Free variables in translated formula')
                visit(f);parsed[kind].append(f)
            except ParseError as error:errors.append(f"{kind}[{item['index']}]: {error}")
    if errors:raise ParseError('; '.join(errors))
    return parsed

class LocalSemanticParser:
    def __init__(self,endpoint='http://127.0.0.1:8765',timeout=240,max_tokens=1800,repair_attempts=2):
        url=urlparse(endpoint)
        if url.scheme!='http' or url.hostname not in ('127.0.0.1','localhost','::1'):
            raise ValueError('Default parser is local only; use a loopback HTTP server')
        self.endpoint=endpoint.rstrip('/');self.timeout=timeout;self.max_tokens=max_tokens;self.repair_attempts=repair_attempts
    def translate(self,context,queries):
        premises=split_premises(context);queries=list(queries)
        payload={'premises':[{'index':i,'text':s} for i,s in enumerate(premises)],'queries':[{'index':i,'text':s} for i,s in enumerate(queries)]}
        messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':json.dumps(payload,ensure_ascii=False)}]
        start=time.perf_counter();repairs=[];attempts=[]
        for attempt in range(self.repair_attempts+1):
            body={'messages':messages,'temperature':0,'seed':2026,'max_tokens':self.max_tokens,
                  'response_format':{'type':'json_schema','json_schema':{'name':'logic_translation','strict':True,'schema':SCHEMA}},
                  'cache_prompt':True}
            request=urllib.request.Request(self.endpoint+'/v1/chat/completions',data=json.dumps(body).encode(),headers={'Content-Type':'application/json'})
            with urllib.request.urlopen(request,timeout=self.timeout) as response:result=json.load(response)
            choice=result['choices'][0];raw=choice['message']['content']
            attempts.append({'raw':raw,'usage':result.get('usage',{})})
            try:
                if choice.get('finish_reason')=='length':raise ParseError('Translation output reached token limit')
                data=json.loads(raw);parsed=validate_translation(data,premises,queries)
                return {'translation':data,'parsed':parsed,'raw':raw,'seconds':time.perf_counter()-start,
                        'usage':result.get('usage',{}),'attempts':attempts,'repairs':repairs,'prompt_version':PROMPT_VERSION,
                        'input_hash':hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()}
            except (ParseError,ValueError,TypeError) as error:
                repairs.append(str(error))
                if attempt==self.repair_attempts:
                    error.details={'raw':raw,'attempts':attempts,'repairs':repairs,'seconds':time.perf_counter()-start}
                    raise
                # Feedback contains syntax/structure errors only, never labels.
                messages=[messages[0],messages[1],{'role':'assistant','content':raw},
                          {'role':'user','content':'The formula validator rejected this translation: '+str(error)+'. Correct the complete JSON while preserving the English meaning. Quantify every variable and preserve every premise. Do not answer the queries.'}]


## Local parser and classical reasoning pipeline

In [10]:
"""Optional language-model translation followed exclusively by classical proofs."""
from collections import OrderedDict,defaultdict
import re
import time


class LocalReasoningAgent(ReasoningAgent):
    """Use local translation for English FOL; keep causal/CWA paths explicit.

    A batch groups identical contexts before translation. No labels, IDs, reference
    rationales or formal annotations are sent to the translator. Failure falls
    back to the deterministic agent with a visible diagnostic, if enabled.
    """
    def __init__(self,mode='owa',timeout=1.0,cache_size=128,endpoint='http://127.0.0.1:8765',
                 reasoner='ground',fallback=True,translator=None,max_queries=8):
        super().__init__(mode,timeout,cache_size)
        if reasoner not in ('ground','resolution','portfolio'):raise ValueError('Unknown reasoner')
        if max_queries<1:raise ValueError('max_queries must be positive')
        self.translator=translator or LocalSemanticParser(endpoint,timeout=360,max_tokens=2200)
        self.reasoner=reasoner;self.fallback=fallback;self.max_queries=max_queries
        self.translation_cache=OrderedDict()

    def clear_cache(self):
        super().clear_cache();self.translation_cache.clear()

    @staticmethod
    def fields(row):
        context=row.get('context',row.get('premises',row.get('given_info',row.get('theory',''))))
        if isinstance(context,list):context='\n'.join(context)
        query=row.get('statement',row.get('question',row.get('conclusion',row.get('query',''))))
        return str(context),str(query)

    def _classical_path(self,context,query,mode):
        return mode=='cwa' or (re.search(r'\bcauses?\b',context,re.I) and re.search(r'\bwould\b.*\boccur\b',query,re.I))

    def _solve_formulas(self,premises,query):
        solver=prove_resolution if self.reasoner=='resolution' else prove_fol
        pred,trace,stats=solver(premises,query,timeout=self.timeout)
        if self.reasoner=='portfolio' and not pred:
            pred,trace2,stats2=prove_resolution(premises,query,timeout=self.timeout)
            trace=trace2 if pred else trace+trace2;stats={'ground':stats,'resolution':stats2}
        return pred,trace,stats

    def _translate_group(self,context,queries):
        start=time.perf_counter();key=(context,tuple(queries));hit=key in self.translation_cache
        try:
            if hit:
                translation=self.translation_cache.pop(key);self.translation_cache[key]=translation
            else:
                translation=self.translator.translate(context,queries)
                # Validate again at the trust boundary, including injected adapters.
                validate_translation(translation['translation'],split_premises(context),queries)
                if self.cache_size>0:
                    self.translation_cache[key]=translation
                    while len(self.translation_cache)>self.cache_size:self.translation_cache.popitem(last=False)
            parsed=validate_translation(translation['translation'],split_premises(context),queries)
        except (OSError,ValueError,TypeError,KeyError,RecursionError) as error:
            parse_seconds=(time.perf_counter()-start)/len(queries);results=[]
            for query in queries:
                r=super().predict(context,query) if self.fallback else Result(False,'local_translation','translation_error')
                r.status='translation_error_fallback' if self.fallback else 'translation_error'
                r.diagnostics.append('Local translator failed: '+str(error));r.parse_seconds+=parse_seconds;r.total_seconds+=parse_seconds
                results.append(r)
            return results
        parse_seconds=(time.perf_counter()-start)/len(queries);results=[]
        for index,query in enumerate(parsed['queries']):
            inference_start=time.perf_counter();diagnostics=[]
            try:
                pred,trace,stats=self._solve_formulas(parsed['premises'],query)
                status='entailed' if pred else 'not_proved'
                if not pred and stats.get('status')=='resource_limit':status='resource_limit'
            except (ParseError,ResourceLimit,RecursionError) as error:
                pred=False;trace=[];stats={};diagnostics=[str(error)]
                status='resource_limit' if isinstance(error,(ResourceLimit,RecursionError)) else 'unsupported_input'
            elapsed=time.perf_counter()-inference_start
            stats.update(translation_cache_hit=hit,prompt_version=translation.get('prompt_version'),
                         translated_premises=translation['translation']['premises'],
                         translated_query=sorted(translation['translation']['queries'],key=lambda x:x['index'])[index],
                         translation_repairs=translation.get('repairs',[]))
            results.append(Result(bool(pred),'local_translation_'+self.reasoner,status,parse_seconds,elapsed,parse_seconds+elapsed,trace,diagnostics,None,stats))
        return results

    def predict(self,context,statement,*,premises_fol=None,conclusion_fol=None,mode=None):
        mode=mode or self.mode
        if self._classical_path(context,statement,mode) or (premises_fol is not None and conclusion_fol is not None):
            return super().predict(context,statement,premises_fol=premises_fol,conclusion_fol=conclusion_fol,mode=mode)
        return self._translate_group(context,[statement])[0]

    def predict_records(self,rows,formal_annotations=False,mode=None):
        mode=mode or self.mode;rows=list(rows);output=[None]*len(rows);groups=defaultdict(list)
        for index,row in enumerate(rows):
            context,query=self.fields(row)
            if self._classical_path(context,query,mode) or (formal_annotations and row.get('premises-FOL') and row.get('conclusion-FOL')):
                output[index]=super().predict_record(row,formal_annotations=formal_annotations,mode=mode)
            else:groups[context].append((index,query))
        for context,items in groups.items():
            for offset in range(0,len(items),self.max_queries):
                batch=items[offset:offset+self.max_queries]
                results=self._translate_group(context,[q for _,q in batch])
                for (index,_),result in zip(batch,results):output[index]=result
        return output


## Syntax-based routing with explicit OWA/CWA inference

In [11]:
"""Syntax-based routing; no dataset identifiers, labels or answer lookups."""
import re


def controlled_english(context,query):
    """Conservative recognition of the small, supported rule grammar."""
    entity=r'(?:the )?[a-z][a-z0-9_]*(?: [a-z][a-z0-9_]*){0,2}'
    property_=r'(?:not )?[a-z][a-z0-9_]*'
    relation=r'(?:(?:does|do) not )?(?:like|likes|visit|visits|see|sees|need|needs|chase|chases|eat|eats|love|loves|help|helps) '+entity
    clause=rf'{entity} (?:is|are) {property_}|{entity} {relation}'
    def part(text,first):
        if not first and re.fullmatch(property_,text):return True
        return bool(re.fullmatch(clause,text))
    def expression(text):return all(part(p,i==0) for i,p in enumerate(text.split(' and ')))
    for sentence in split_premises(context)+[query]:
        text=clean_nl(sentence)
        match=re.fullmatch(r'if (.+?)(?:,? then |, )(.+)',text)
        if not match:match=re.fullmatch(r'(?:given that|when) (.+?), (?:it follows that |then )?(.+)',text)
        if match:
            if not expression(match[1]) or not expression(match[2]):return False
        elif re.fullmatch(r'all (?:not )?\w+ (?:things|people) are (?:not )?\w+',text):continue
        elif not expression(text):return False
    return True


class AdaptiveReasoningAgent(LocalReasoningAgent):
    """Translate complex English, solve CWA and OWA with their own semantics.

    Controlled CWA and structural causal inputs use the deterministic engine.
    Translation is optional and its errors/fallbacks stay visible in Result.
    """
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        if self.mode=='cwa':self.max_queries=24

    def _classical_path(self,context,query,mode):
        causal=re.search(r'\bcauses?\b',context,re.I) and re.search(r'\bwould\b.*\boccur\b',query,re.I)
        return bool(causal or (mode=='cwa' and controlled_english(context,query)))

    def _solve_formulas(self,premises,query):
        if self.mode!='cwa':return super()._solve_formulas(premises,query)
        constants=set()
        def collect(f):
            if f[0]=='atom':constants.update(t for t in f[2] if not variable(t))
            elif f[0] in ('all','exists'):collect(f[2])
            else:
                for item in f[1:]:collect(item)
        for formula in premises+[query]:collect(formula)
        key=('translated_cwa',tuple(premises),tuple(sorted(constants,key=repr)))
        hit=key in self.cache
        if hit:model=self.cache.pop(key);self.cache[key]=model
        else:
            model=HornModel(premises,constants,'cwa',self.timeout)
            if self.cache_size>0:
                self.cache[key]=model
                while len(self.cache)>self.cache_size:self.cache.popitem(last=False)
        prediction=model.holds(query)
        return prediction,model.explain(query),{'facts':len(model.model),'rules':len(model.rules),
                   'rounds':model.rounds,'cache_hit':hit,'semantics':'cwa','well_founded':getattr(model,'well_founded',False)}


## Regression tests and independent solver checks
The tests cover failures in the supplied prototypes and compare DPLL with exhaustive truth tables.

In [12]:
import unittest, random, itertools
class LanguageTests(unittest.TestCase):
    def expect(self,context,query,expected,mode='owa'):
        r=ReasoningAgent(mode).predict(context,query)
        self.assertEqual(r.prediction,expected,r)
        self.assertNotEqual(r.status,'unsupported_input',r)
    def test_assignment_verbatim(self):
        self.expect('Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.','Bob is successful.',True)
    def test_negative_fact(self):self.expect('Bob is not kind.','Bob is kind.',False)
    def test_negative_query(self):self.expect('Bob is kind.','Bob is not kind.',False)
    def test_negative_fact_proof(self):self.expect('Bob is not kind.','Bob is not kind.',True)
    def test_negation_not_closed_world(self):self.expect('Bob is young.','Bob is not kind.',False)
    def test_tall_prefix(self):self.expect('Bob is tall. If someone is tall then they are kind.','Bob is kind.',True)
    def test_article_prefix(self):self.expect('Bob is an artist. If someone is an artist then they are kind.','Bob is kind.',True)
    def test_whole_query(self):self.expect('Bob is kind.','Bob is kind and smart.',False)
    def test_universal_fact(self):self.expect('Everything is blue.','Bob is blue.',True)
    def test_universal_rule(self):self.expect('Bob is rough. All rough things are quiet.','Bob is quiet.',True)
    def test_verbal_universal(self):self.expect('Tom is a citizen of Lawton Park. All citizens of Lawton Park use the zip code 98199.','Tom uses the zip code 98199.',True)
    def test_relation_inflections(self):self.expect('The cat sees the dog. If someone sees the dog then they like the dog.','The cat likes the dog.',True)
    def test_second_variable(self):self.expect('Alice likes Bob. If someone likes something then they see it.','Alice sees Bob.',True)
    def test_second_variable_condition(self):self.expect('Alice likes Bob. Bob is red. If someone likes something and it is red then they see it.','Alice sees Bob.',True)
    def test_headings_and_paraphrases(self):
        for rule in ['If Bob is young, Bob is smart.','Assuming Bob is young, we can say Bob is smart.','When Bob is young, we can conclude Bob is smart.','Given that Bob is young, it follows that Bob is smart.','Based on Bob is young, it is evident that Bob is smart.']:
            with self.subTest(rule=rule):self.expect('Facts:\nBob is young.\nRules:\n'+rule,'Bob is smart.',True)
    def test_multiword_subject_order(self):
        for ctx in ['Stanford University cooks well. Stanford University writes stories.','Stanford University writes stories. Stanford University cooks well.']:
            self.expect(ctx+' If Stanford University cooks well then Bob is kind.','Bob is kind.',True)
    def test_empty_kb(self):self.expect('','Bob is kind.',False)
    def test_positive_cycle(self):self.expect('If someone is young then they are kind. If someone is kind then they are young.','Bob is kind.',False)
    def test_explicit_negative_head(self):self.expect('Bob is young. If someone is young then they are not kind.','Bob is not kind.',True)
    def test_cwa_strata(self):self.expect('Bob is kind. If someone is kind and not red then they are blue. If someone is kind then they are red.','Bob is blue.',False,'cwa')
    def test_cwa_cycle_rejected(self):
        r=ReasoningAgent('cwa').predict('Bob is kind. If someone is kind and not red then they are red.','Bob is red.')
        self.assertEqual(r.status,'unsupported_input')
    def test_malformed_rule_does_not_drop_condition(self):
        r=ReasoningAgent().predict('If Bob is young ?! invalid ???','Bob is smart.')
        self.assertFalse(r.prediction)
    def test_cache_preserves_result(self):
        a=ReasoningAgent();r1=a.predict('Bob is kind.','Bob is kind.');r2=a.predict('Bob is kind.','Bob is kind.')
        self.assertEqual(r1.prediction,r2.prediction);self.assertTrue(r2.stats['cache_hit'])
    def test_labels_and_reference_reasoning_ignored(self):
        a=ReasoningAgent()
        row={'context':'Bob is kind.','statement':'Bob is smart.','flag':'True','reasoning':'The answer is True'}
        first=a.predict_record(row).prediction
        row.update(flag='False',reasoning='The answer is False')
        self.assertFalse(first);self.assertEqual(first,a.predict_record(row).prediction)
    def test_negative_head_has_distinct_stratum(self):
        self.expect('Bob is kind. If someone is kind and not red then they are not kind.','Bob is not red.',True,'cwa')
    def test_relative_clause_preserves_restriction(self):
        context='All artists who like music are friendly. Nina likes music.'
        self.expect(context,'Nina is friendly.',False)
        self.expect(context+' Nina is an artist.','Nina is friendly.',True)
    def test_everyone_relative(self):
        self.expect('Everyone who knows algebra likes puzzles. Mira knows algebra.','Mira likes puzzles.',True)
    def test_relative_clause_in_consequent(self):
        self.expect('All dogs are animals that bark. Milo is a dog.','Milo barks.',True)
    def test_coordinated_named_subjects(self):
        self.expect('Ada and Lin are musicians. All musicians are artists.','Ada is an artist and Lin is an artist.',True)
    def test_implicit_generic_plural(self):
        self.expect('Birds are animals. Robin is a bird.','Robin is an animal.',True)
    def test_scoped_not_all(self):
        self.expect('Some birds are not blue.','Not all birds are blue.',True)
    def test_universal_query_needs_all_cases(self):
        self.expect('Mira is an artist. Mira is happy. Leo is an artist. Leo is not happy.','If someone is an artist, then they are happy.',False)
    def test_contraposition_in_categorical_english(self):
        self.expect('All painters are artists. Nora is not an artist.','Nora is not a painter.',True)
    def test_either_inclusive(self):
        self.expect('Nora is either blue or red. Nora is blue.','Nora is not red.',False)
    def test_explicit_exclusive_or(self):
        self.expect('Nora is either blue or red, but not both. Nora is blue.','Nora is not red.',True)
    def test_elliptical_neither(self):
        self.expect('No artists are pilots. Nora is either both an artist and a pilot, or neither.','Nora is neither an artist nor a pilot.',True)
    def test_negated_shared_disjunction(self):
        self.expect('Nora does not sing or dance.','Nora does not dance.',True)
    def test_progressive_and_possessive(self):
        self.expect('If someone works in a lab, then they use their hands. Nora is working in a lab.','Nora uses her hands.',True)
    def test_existential_query(self):
        self.expect('Mira is a scientist. Mira likes music.','A scientist likes music.',True)
    def test_exhaustive_category_list(self):
        self.expect('There are three types of badges: gold badges, silver badges, and bronze badges. This badge is a badge. This badge is neither a gold badge nor a silver badge.','This badge is a bronze badge.',True)
    def test_named_list(self):
        self.expect('A list of students at the academy: Mira, Nora, and Lin.','Nora is a student at the academy.',True)
    def test_intersective_modifier(self):
        self.expect('Mira is a famous artist.','Mira is an artist.',True)
    def test_nonintersective_modifier(self):
        self.expect('This object is a fake diamond.','This object is a diamond.',False)
    def test_query_cache_isolation(self):
        a=ReasoningAgent();context='Mira is a famous artist.'
        self.assertTrue(a.predict(context,'Mira is an artist.').prediction)
        self.assertFalse(a.predict(context,'Mira is a scientist.').prediction)
        self.assertTrue(a.predict(context,'Mira is famous.').prediction)
        self.assertEqual(len(a.parse_cache),1);a.clear_cache();self.assertFalse(a.parse_cache)
    def test_partial_language_premise(self):
        r=ReasoningAgent().predict('Mira is kind. If malformed.','Mira is kind.')
        self.assertTrue(r.prediction);self.assertEqual(r.status,'entailed_partial_input')

class FormalTests(unittest.TestCase):
    def expect(self,p,q,expected):
        r=ReasoningAgent(timeout=2).predict('','',premises_fol=p,conclusion_fol=q)
        self.assertEqual(r.prediction,expected,r)
        self.assertNotIn(r.status,('resource_limit','unsupported_input'),r)
    def test_universal(self):self.expect('∀x (P(x) → Q(x))\nP(alice)','Q(alice)',True)
    def test_disjunctive_head(self):self.expect('P(alice)\nP(alice) → Q(alice) ∨ R(alice)','Q(alice)',False)
    def test_disjunction_resolution(self):self.expect('P(alice) ∨ Q(alice)\n¬P(alice)','Q(alice)',True)
    def test_antecedent_preserved(self):self.expect('(P(alice) ∧ R(alice)) → Q(alice)','Q(alice)',False)
    def test_exists_independent(self):self.expect('∃x P(x)\n∃x Q(x)\n∀x (P(x) ∧ Q(x) → R(alice))','R(alice)',False)
    def test_existential_query(self):self.expect('P(alice)','∃x P(x)',True)
    def test_universal_query(self):self.expect('∀x P(x)','∀y P(y)',True)
    def test_no_unique_names(self):self.expect('P(alice)\nQ(bob)','¬(alice=bob)',False)
    def test_equality_congruence(self):self.expect('alice=bob\nP(alice)','P(bob)',True)
    def test_equality_reflexive(self):self.expect('P(alice)','alice=alice',True)
    def test_whole_query(self):self.expect('P(alice)','P(alice) ∧ Q(alice)',False)
    def test_unknown_not_negative(self):self.expect('P(alice)','¬Q(alice)',False)
    def test_alternative_bindings(self):self.expect('Edge(alice,bob)\nEdge(alice,dan)\nEdge(dan,carol)\n∀x ∀y ∀z (Edge(x,y) ∧ Edge(y,z) → Path(x,z))','Path(alice,carol)',True)
    def test_nested_quantifiers(self):self.expect('∀x (P(x) → ∃y R(x,y))\nP(alice)','∃z R(alice,z)',True)
    def test_iff_and_xor(self):self.expect('P(a) ↔ Q(a)\nP(a) ⊕ R(a)\n¬R(a)','Q(a)',True)
    def test_contraposition(self):self.expect('∀x (P(x) → Q(x))\n¬Q(a)','¬P(a)',True)
    def test_partial_premise_has_diagnostic(self):
        r=ReasoningAgent().predict('','',premises_fol='P(a)\nUnsupported(:text)',conclusion_fol='P(a)')
        self.assertTrue(r.prediction);self.assertEqual(r.status,'entailed_partial_input');self.assertTrue(r.diagnostics)
    def test_occurs_check(self):self.assertIsNone(unify_term('?x',('fn','f',('?x',)),{}))
    def test_unicode_and_numeric_identifiers(self):self.expect('P(Łódź)\n∀x (P(x) → Visits(x,2027Event))','Visits(Łódź,2027Event)',True)
    def test_long_xor_does_not_hide_simple_proof(self):
        names=[f'P{i}(a)' for i in range(16)]
        self.expect(' ⊕ '.join(names)+'\nQ(a)','Q(a)',True)
    def test_lifted_unification_keeps_alternative_witnesses(self):
        self.expect('∀x (P(x) → ∃y R(x,y))\nP(a)\n∀x ∀y (R(x,y) → Q(x))','Q(a)',True)
        self.expect('∀x (P(x) → ∃y R(x,y))\nP(a)','R(a,b)',False)

class CausalTests(unittest.TestCase):
    def expect(self,c,q,p):
        r=ReasoningAgent().predict(c,q);self.assertEqual(r.prediction,p,r)
        self.assertNotEqual(r.status,'unsupported_input',r)
    def test_chain(self):self.expect('A causes B. B causes C.','Would C occur if A instead of not A?',True)
    def test_child_list(self):self.expect('A causes B, C, and D.','Would C occur if A instead of not A?',True)
    def test_or_together(self):self.expect('A or B together cause C. We observed B.','Would C occur if not A instead of A?',True)
    def test_and_without_together(self):self.expect('A and B causes C. We observed B.','Would C occur if not A instead of A?',False)
    def test_abduction(self):self.expect('B causes A, A or B causes C, C causes D. B~Bern(0.2). We observed C.','Would D occur if not A instead of A?',True)
    def test_recompute_observed_child(self):self.expect('B causes A, A and B causes C, C causes D. B~Bern(0.2). We observed C.','Would D occur if not A instead of A?',False)
    def test_negative_observation_prefix(self):self.assertEqual(CausalModel('A causes B. We observed not A.').observed,{'a':False})
    def test_negative_observation_suffix(self):self.assertEqual(CausalModel('A causes B. We observed A not occurs.').observed,{'a':False})
    def test_nested_actions(self):self.expect('A causes B, B causes C.','Assume A, and further suppose not B. Would C occur?',False)
    def test_cycle_diagnostic(self):
        r=ReasoningAgent().predict('A causes B, B causes A.','Would B occur if A instead of not A?');self.assertEqual(r.status,'unsupported_input')

class SolverCrossChecks(unittest.TestCase):
    def test_full_refutation_against_boolean_models(self):
        import random
        rng=random.Random(2077);letters=[atom(x) for x in ('p','q','r')]
        def expression(depth):
            if depth==0:return rng.choice(letters)
            op=rng.choice(('not','and','or','implies','iff','xor'))
            return (op,expression(depth-1)) if op=='not' else (op,expression(depth-1),expression(depth-1))
        def truth(f,values):
            if f[0]=='atom':return values[f[1]]
            if f[0]=='not':return not truth(f[1],values)
            a,b=(truth(x,values) for x in f[1:])
            return {'and':a and b,'or':a or b,'implies':not a or b,'iff':a==b,'xor':a!=b}[f[0]]
        for _ in range(40):
            premises=[expression(2),expression(2)];query=expression(2)
            expected=all(not all(truth(f,dict(zip(('p','q','r'),bits))) for f in premises) or truth(query,dict(zip(('p','q','r'),bits))) for bits in itertools.product((False,True),repeat=3))
            actual,_,_=prove_fol(premises,query,timeout=2)
            self.assertEqual(actual,expected,(premises,query))
    def test_dpll_against_truth_table(self):
        # Exhaustive 3-variable truth table is independent of DPLL's internals.
        candidates=[frozenset(x) for n in (1,2,3) for x in itertools.combinations((-3,-2,-1,1,2,3),n) if not any(-v in x for v in x)]
        import random
        rng=random.Random(2026)
        for _ in range(150):
            clauses=rng.sample(candidates,rng.randrange(1,9))
            expected=any(all(any(bits[abs(l)-1]==(l>0) for l in c) for c in clauses) for bits in itertools.product((False,True),repeat=3))
            self.assertEqual(dpll(clauses,time.perf_counter()+1,{'nodes':0,'units':0}),expected)


class ResolutionTests(unittest.TestCase):
    def check(self,premises,query,expected):
        p=[FormulaParser().parse(f) for f in premises.splitlines()]
        actual,trace,stats=prove_resolution(p,FormulaParser().parse(query),timeout=2)
        self.assertEqual(actual,expected,(trace,stats))
    def test_chain(self):self.check('P(a)\n∀x (P(x) → Q(x))\n∀x (Q(x) → R(x))','R(a)',True)
    def test_proof_by_cases(self):self.check('P(a) ∨ Q(a)\n∀x (P(x) → R(x))\n∀x (Q(x) → R(x))','R(a)',True)
    def test_contraposition(self):self.check('∀x (P(x) → Q(x))\n¬Q(a)','¬P(a)',True)
    def test_factoring(self):self.check('∀x ∀y (P(x) ∨ P(y))\n∀x ∀y (¬P(x) ∨ ¬P(y))','Q(a)',True)
    def test_standardize_apart(self):self.check('∀x (P(x) → Q(x))\nP(a)\nP(b)','Q(a) ∧ Q(b)',True)
    def test_existential_not_named_witness(self):self.check('∃x P(x)','P(a)',False)
    def test_independent_witnesses(self):self.check('∃x P(x)\n∃x Q(x)','∃x (P(x) ∧ Q(x))',False)
    def test_function_witness(self):self.check('∀x (P(x) → ∃y R(x,y))\nP(a)','∃y R(a,y)',True)
    def test_universal_existential(self):self.check('∀x ∃y R(x,y)\n∀x ∀y (R(x,y) → P(y))','∃x P(x)',True)
    def test_equality(self):self.check('a=b\nP(a)','P(b)',True)
    def test_function_congruence(self):self.check('a=b','f(a)=f(b)',True)
    def test_no_unique_names(self):self.check('P(a)\nQ(b)','¬(a=b)',False)
    def test_unknown_is_not_negative(self):self.check('P(a)','¬Q(a)',False)
    def test_disjunction_not_conjunction(self):self.check('P(a) ∨ Q(a)','P(a) ∧ Q(a)',False)
    def test_resource_limit_never_proves(self):
        p=[FormulaParser().parse('P(a)')];q=FormulaParser().parse('Q(a)')
        result,_,stats=prove_resolution(p,q,max_clauses=1)
        self.assertFalse(result);self.assertEqual(stats['status'],'resource_limit')
    def test_truth_table_crosscheck(self):
        rng=random.Random(814);letters=[atom(x) for x in ('p','q','r')]
        def make(d):
            if d==0:return rng.choice(letters)
            op=rng.choice(('not','and','or','implies','iff','xor'))
            return (op,make(d-1)) if op=='not' else (op,make(d-1),make(d-1))
        def truth(f,vals):
            if f[0]=='atom':return vals[f[1]]
            if f[0]=='not':return not truth(f[1],vals)
            a,b=[truth(x,vals) for x in f[1:]]
            return {'and':a and b,'or':a or b,'implies':not a or b,'iff':a==b,'xor':a!=b}[f[0]]
        for _ in range(40):
            p=[make(2),make(2)];q=make(2)
            expected=all(not all(truth(f,dict(zip(('p','q','r'),bits))) for f in p) or truth(q,dict(zip(('p','q','r'),bits))) for bits in itertools.product((False,True),repeat=3))
            actual,trace,stats=prove_resolution(p,q,timeout=2)
            self.assertEqual(actual,expected,(p,q,trace,stats))

class SemanticParserTests(unittest.TestCase):
    def valid(self):return {'premises':[{'index':0,'formula':'∀x (P(x) → Q(x))'},{'index':1,'formula':'P(alice)'}],'queries':[{'index':0,'formula':'Q(alice)'}]}
    def check_bad(self,data):
        with self.assertRaises((ParseError,ValueError,TypeError)):validate_translation(data,['one','two'],['query'])
    def test_well_formed(self):self.assertEqual(len(validate_translation(self.valid(),['one','two'],['query'])['premises']),2)
    def test_reject_answer_label(self):
        data=self.valid();data['answer']=True;self.check_bad(data)
    def test_reject_missing_premise(self):
        data=self.valid();data['premises'].pop();self.check_bad(data)
    def test_reject_duplicate_source(self):
        data=self.valid();data['premises'][1]['index']=0;self.check_bad(data)
    def test_reject_inconsistent_arity(self):
        data=self.valid();data['queries'][0]['formula']='Q(alice,bob)';self.check_bad(data)
    def test_reject_unbound_variable(self):
        data=self.valid();data['queries'][0]['formula']='Q(x)';self.check_bad(data)
    def test_reject_unbound_variable_in_function(self):
        data=self.valid();data['queries'][0]['formula']='Q(f(x))';self.check_bad(data)
    def test_reports_multiple_bad_formulas(self):
        data=self.valid();data['premises'][0]['formula']='P(x)';data['queries'][0]['formula']='Q(y)'
        with self.assertRaises(ParseError) as caught:validate_translation(data,['one','two'],['query'])
        self.assertIn('premises[0]',str(caught.exception));self.assertIn('queries[0]',str(caught.exception))
    def test_reject_executable_text(self):
        data=self.valid();data['queries'][0]['formula']='Q(alice); import os';self.check_bad(data)
    def test_reject_remote_endpoint(self):
        with self.assertRaises(ValueError):LocalSemanticParser('https://example.org')
    def test_preserve_explicit_lines(self):self.assertEqual(split_premises('Dr. Mara is kind.\nMara is an artist.'),['Dr. Mara is kind.','Mara is an artist.'])

class LocalPipelineTests(unittest.TestCase):
    class Translator:
        def __init__(self):self.calls=[]
        def translate(self,context,queries):
            self.calls.append((context,queries))
            return {'translation':{'premises':[{'index':0,'formula':'P(alice)'}],
                                  'queries':[{'index':i,'formula':'P(alice)' if q=='Known?' else 'Q(alice)'} for i,q in enumerate(queries)]}}
    def test_groups_contexts_and_preserves_order(self):
        parser=self.Translator();agent=LocalReasoningAgent(translator=parser)
        rows=[{'context':c,'statement':q,'label':'secret gold','reasoning':'secret rationale'} for c,q in [('A.','Known?'),('B.','Unknown?'),('A.','Unknown?')]]
        self.assertEqual([r.prediction for r in agent.predict_records(rows)],[True,False,False])
        self.assertEqual(parser.calls,[('A.',['Known?','Unknown?']),('B.',['Unknown?'])])
        self.assertNotIn('secret',repr(parser.calls))
    def test_cache_is_query_sensitive(self):
        parser=self.Translator();agent=LocalReasoningAgent(translator=parser)
        self.assertTrue(agent.predict('A.','Known?').prediction)
        self.assertFalse(agent.predict('A.','Unknown?').prediction)
        self.assertTrue(agent.predict('A.','Known?').stats['translation_cache_hit'])
        self.assertEqual(len(parser.calls),2)
    def test_chunk_limit(self):
        parser=self.Translator();agent=LocalReasoningAgent(translator=parser,max_queries=2)
        rows=[{'context':'A.','statement':str(i)} for i in range(5)]
        self.assertEqual(len(agent.predict_records(rows)),5)
        self.assertEqual([len(qs) for _,qs in parser.calls],[2,2,1])
    def test_offline_fallback_is_visible(self):
        class Broken:
            def translate(self,*args):raise OSError('offline')
        r=LocalReasoningAgent(translator=Broken()).predict('Nora is kind.','Nora is kind.')
        self.assertTrue(r.prediction);self.assertEqual(r.status,'translation_error_fallback')
        self.assertIn('offline',r.diagnostics[-1]);self.assertGreater(r.total_seconds,0)
    def test_malformed_translation_cannot_prove(self):
        class Bad:
            def translate(self,*args):return {'translation':{'answer':True}}
        r=LocalReasoningAgent(translator=Bad(),fallback=False).predict('A.','Known?')
        self.assertFalse(r.prediction);self.assertEqual(r.status,'translation_error')
    def test_causal_and_cwa_bypass_translator(self):
        parser=self.Translator();agent=LocalReasoningAgent(translator=parser)
        self.assertFalse(agent.predict('A causes B.','Would B occur if not A instead of A?').prediction)
        self.assertTrue(agent.predict('Nora is kind.','Nora is kind.',mode='cwa').prediction)
        self.assertEqual(parser.calls,[])


class GeneralizationTests(unittest.TestCase):
    def test_restated_definite_subject(self):
        a=ReasoningAgent('cwa')
        c='The swift fox likes the owl. The owl needs the fox. If the swift fox likes the owl and the owl needs the fox then the swift fox is cheerful.'
        self.assertTrue(a.predict(c,'The swift fox is cheerful.').prediction)
    def test_generic_pronoun_survives_named_conjunct(self):
        c='The hare likes the elk. The elk is red. If someone likes the elk and the elk is red then they are warm.'
        a=ReasoningAgent('cwa');self.assertTrue(a.predict(c,'The hare is warm.').prediction)
        self.assertFalse(a.predict(c,'The elk is warm.').prediction)
    def test_negation_cycle_resolved_by_fact(self):
        a=ReasoningAgent('cwa');c='Ada is red. If Ada is not blue then Ada is red. If Ada is not red then Ada is blue.'
        self.assertTrue(a.predict(c,'Ada is red.').prediction)
        self.assertTrue(a.predict(c,'Ada is not blue.').prediction)
    def test_undefined_cycle_does_not_invalidate_unrelated_fact(self):
        a=ReasoningAgent('cwa');c='Lia is tall. If Ada is not red then Ada is red.'
        self.assertTrue(a.predict(c,'Lia is tall.').prediction)
        self.assertEqual(a.predict(c,'Ada is red.').status,'unsupported_input')
        self.assertEqual(a.predict(c,'Ada is not red.').status,'unsupported_input')
    def test_no_global_closed_world_for_folio(self):
        self.assertFalse(ReasoningAgent('owa').predict('Lia is tall.','Lia is not blue.').prediction)
    def test_partial_cwa_input_cannot_prove_negative(self):
        r=ReasoningAgent('cwa').predict('If this condition is malformed','Lia is not blue.')
        self.assertFalse(r.prediction);self.assertEqual(r.status,'unsupported_input')
    def test_partial_cwa_input_can_use_sound_positive_support(self):
        r=ReasoningAgent('cwa').predict('Ada is red. If malformed ??','Ada is red.')
        self.assertTrue(r.prediction);self.assertEqual(r.status,'entailed_partial_input')
    def test_causal_cache_does_not_reuse_intervention(self):
        a=ReasoningAgent();c='A causes B, B causes C.'
        first=a.predict(c,'Would C occur if A?');second=a.predict(c,'Would C occur if not A?')
        self.assertTrue(first.prediction);self.assertFalse(second.prediction);self.assertTrue(second.stats['cache_hit'])
        a.clear_cache();self.assertFalse(a.predict(c,'Would C occur if not A?').stats['cache_hit'])
    def test_router_requires_known_grammar(self):
        self.assertTrue(controlled_english('Neris is warm. If Neris is warm then Neris is calm.','Neris is calm.'))
        self.assertFalse(controlled_english('Neris seems extremely calm.','Neris is calm.'))
        self.assertFalse(controlled_english('Neris is green with envy.','Neris is green.'))
    def test_translated_cwa_preserves_negation_semantics(self):
        a=AdaptiveReasoningAgent(mode='cwa',translator=object())
        p=atom('p','nora');q=atom('q','nora')
        self.assertTrue(a._solve_formulas([p],neg(q))[0])
        self.assertFalse(a._solve_formulas([p],q)[0])
    def test_well_founded_against_exhaustive_stable_models(self):
        rng=random.Random(723);atoms=[atom(p,'n') for p in 'abcd'];literals=[(True,a) for a in atoms]
        for _ in range(100):
            facts={a for a in atoms if rng.random()<.2};rules=[]
            for j in range(6):
                head=rng.choice(atoms);positive=set(rng.sample(atoms,rng.randrange(3)));negative=set(rng.sample(atoms,rng.randrange(3)))
                rules.append((positive,negative,head))
            stable=[]
            for bits in itertools.product((False,True),repeat=4):
                assumed={a for a,b in zip(atoms,bits) if b};closure=set(facts)
                while True:
                    added={h for pos,negative,h in rules if not negative&assumed and pos<=closure}
                    if added<=closure:break
                    closure|=added
                if closure==assumed:stable.append(assumed)
            formulas=list(facts)+[('implies',('and',*sorted(pos,key=repr),*(neg(a) for a in sorted(negative,key=repr))),h) for pos,negative,h in rules]
            model=HornModel(formulas,{'n'},'cwa',timeout=2)
            if stable:
                true={a for sign,a in model.model if sign};unknown={a for sign,a in getattr(model,'undefined',set()) if sign}
                self.assertTrue(true<=set.intersection(*stable))
                self.assertTrue(set.union(*stable)<=true|unknown)


suite = unittest.TestSuite()
for test_class in (LanguageTests, FormalTests, CausalTests, SolverCrossChecks, ResolutionTests, SemanticParserTests, LocalPipelineTests, GeneralizationTests):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(test_class))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
assert test_result.wasSuccessful(), 'Regression tests failed'
print(f'Regression tests: {test_result.testsRun} passed.')


Regression tests: 122 passed.


..........................................................................................................................
----------------------------------------------------------------------
Ran 122 tests in 1.048s

OK


## Reasoning traces
Each trace below is generated by the executing code.

In [13]:

agent = ReasoningAgent()
context = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful."
examples = [
    agent.predict(context, "Bob is successful."),
    agent.predict("", "", premises_fol="∀x (P(x) → Q(x))\nP(alice)", conclusion_fol="Q(alice)"),
    agent.predict("B causes A, A or B causes C, C causes D. B~Bern(0.2). We observed C.", "Would D occur if not A instead of A?")
]
for result in examples:
    print(f"\nPrediction={result.prediction}, method={result.method}, status={result.status}")
    print("\n".join(result.trace))



Prediction=True, method=horn_owa, status=entailed
successful(bob) <= ((smart(?x) and kind(?x)) implies successful(?x))
  smart(bob) <= (young(?x) implies smart(?x))
    young(bob) <= fact
  kind(bob) <= fact

Prediction=True, method=fol_ground_refutation, status=entailed
Lifted unit resolution refuted KB and negated query (3 unifications, 6 clauses).

Prediction=True, method=causal_enumeration, status=entailed
Abduction: 1 exogenous assignments consistent with observations {'c': True}.
Action: do({'a': False}); replace intervened equations.
Deduction: recompute endogenous variables in topological order.
P(d=True | observations, intervention) = 1.000000.
Binary causal decision uses probability > 0.5; exact probability is returned separately.


## Offline public dataset snapshots

Sources: [jise/ruletaker](https://huggingface.co/datasets/jise/ruletaker),
[tasksource/ruletaker](https://huggingface.co/datasets/tasksource/ruletaker),
[CounterBench](https://huggingface.co/datasets/CounterBench/CounterBench),
and [tasksource/folio](https://huggingface.co/datasets/tasksource/folio).
The assignment's Yale FOLIO link is gated; the public mirror is identified explicitly.

The compressed JSON below contains **public examples**, not a model or memorized
prediction table. Inference has no access to the `flag`, `label`, `answer`, `reasoning`,
or metadata fields. Downloading is excluded from inference timings. The full project
also includes development data and a downloader with source URLs and checksums.


In [14]:
import base64, zlib, hashlib
PUBLIC_DATA_B64 = '...'
public_json = zlib.decompress(base64.b64decode(PUBLIC_DATA_B64))
assert hashlib.sha256(public_json).hexdigest() == 'b1f7588663fcea08156ef9c89435014cc60b1c5c0bff7af03c7bd5e16503c32a'
PUBLIC_DATA = json.loads(public_json)
print({name: len(rows) for name, rows in PUBLIC_DATA.items()})


{'jise_test': 250, 'jise_ood_test': 375, 'tasksource_dev200': 200, 'folio_validation': 203, 'counterbench_alpha': 1000, 'counterbench_backdoor': 200}


## Generalization audit: frozen public split and untouched evaluation

The full project contains the reproducible grouped split manifest, source hashes,
prediction files, error analysis and evaluator. This cell shows the measured results
from that audit; it does not use the private test set. RuleTaker uses official
original English with matching released Problog annotations. FOLIO stories are
grouped by logical template and near-duplicate text before allocation. CounterBench
has no untouched public split and was previously examined, so it is reported only
as validation/regression. The final fresh estimate covers RuleTaker and FOLIO.


In [15]:
GENERALIZATION_AUDIT = json.loads('{"partitions": {"validation": {"n": 2602, "labels": {"True": 1265, "False": 1337}, "datasets": {"RuleTaker": 1748, "FOLIO": 294, "CounterBench": 560}}, "final_fresh": {"n": 2938, "labels": {"True": 1434, "False": 1504}, "datasets": {"RuleTaker": 2616, "FOLIO": 322}}, "final_legacy": {"n": 640, "labels": {"True": 320, "False": 320}, "datasets": {"CounterBench": 640}}, "training_reserved": {"n": 176, "labels": {"True": 68, "False": 108}, "datasets": {"FOLIO": 176}}, "diagnostics_validation": {"n": 96, "labels": {"True": 48, "False": 48}, "datasets": {"diagnostic": 96}}, "diagnostics_final": {"n": 96, "labels": {"True": 48, "False": 48}, "datasets": {"diagnostic": 96}}}, "baseline_validation": {"n": 2602, "correct": 2164, "accuracy": 83.16679477325134, "precision": 0.9241025641025641, "recall": 0.7122529644268775, "f1": 0.8044642857142857, "confusion_matrix": [[1263, 74], [364, 901]], "labels": {"True": 1265, "False": 1337}, "raw_labels": {"True": 990, "False": 955, "Uncertain": 97, "no": 285, "yes": 275}, "parsing_failure_count": 229, "unsupported_count": 474, "statuses": {"entailed": 923, "not_proved": 866, "unsupported_input": 474, "entailed_partial_input": 52, "resource_limit": 7, "not_entailed": 280}, "independent_groups": 360, "error_categories": {"none": 2069, "undefined_under_cwa": 176, "unsupported_rule_representation": 120, "parsing_failure": 229, "unsupported_input": 1, "resource_limit": 7}, "inference_seconds": {"mean": 0.00014353935438787496, "median": 2.2649999664281495e-05, "p95": 0.0005868200014447208, "sum": 0.3734894001172506}, "response_seconds": {"mean": 0.003420228093830902, "median": 0.0033189999994647224, "p95": 0.007837299999664538, "sum": 8.899433500148007}, "amortized_seconds": {"mean": 0.00039706656418392534, "median": 0.0002745562501331733, "p95": 0.0009822250003708177, "sum": 1.0331672000065737}, "parse_seconds": {"mean": 0.0002469682551710446, "median": 0.00010435000149300322, "p95": 0.00123391500001162, "sum": 0.642611399955058}, "parsing_failure_rate": 0.08800922367409685, "unsupported_rate": 0.18216756341275941}, "improved_validation": {"n": 2602, "correct": 2279, "accuracy": 87.58647194465796, "precision": 0.9635826771653543, "recall": 0.7739130434782608, "f1": 0.8583954405962297, "confusion_matrix": [[1300, 37], [286, 979]], "labels": {"True": 1265, "False": 1337}, "raw_labels": {"True": 990, "False": 955, "Uncertain": 97, "no": 285, "yes": 275}, "parsing_failure_count": 229, "unsupported_count": 345, "statuses": {"entailed": 1011, "not_proved": 954, "unsupported_input": 345, "entailed_partial_input": 5, "resource_limit": 7, "not_entailed": 280}, "independent_groups": 360, "error_categories": {"none": 2245, "unsupported_rule_representation": 120, "parsing_failure": 229, "unsupported_input": 1, "resource_limit": 7}, "inference_seconds": {"mean": 0.00024487863182119657, "median": 1.950000000761065e-05, "p95": 0.0010052750000795639, "sum": 0.6371741999987535}, "response_seconds": {"mean": 0.005196570830125519, "median": 0.005247699999927136, "p95": 0.010678099999950064, "sum": 13.5214772999866}, "amortized_seconds": {"mean": 0.0006480530745587669, "median": 0.0004129624999933412, "p95": 0.0016795000000608222, "sum": 1.6862341000019114}, "parse_seconds": {"mean": 0.0003927267486530108, "median": 0.00015580000012960227, "p95": 0.0020817199999783034, "sum": 1.0218749999951342}, "parsing_failure_rate": 0.08800922367409685, "unsupported_rate": 0.13259031514219832}, "baseline_final": {"n": 2938, "correct": 2332, "accuracy": 79.37372362151123, "precision": 0.8709677419354839, "recall": 0.6778242677824268, "f1": 0.7623529411764706, "confusion_matrix": [[1360, 144], [462, 972]], "labels": {"True": 1434, "False": 1504}, "raw_labels": {"True": 1434, "False": 1394, "Uncertain": 110}, "parsing_failure_count": 435, "unsupported_count": 606, "statuses": {"entailed": 961, "not_proved": 1205, "unsupported_input": 606, "entailed_partial_input": 155, "resource_limit": 11}, "independent_groups": 320, "error_categories": {"none": 2166, "undefined_under_cwa": 246, "parsing_failure": 435, "unsupported_rule_representation": 80, "resource_limit": 11}, "inference_seconds": {"mean": 0.00027451657590237997, "median": 1.3799999976527033e-05, "p95": 0.0010892999999896347, "sum": 0.8065297000011924}, "response_seconds": {"mean": 0.007839869128657122, "median": 0.006955000000061773, "p95": 0.015540500000042812, "sum": 23.033535499994628}, "amortized_seconds": {"mean": 0.0007587017699112145, "median": 0.000494935000000396, "p95": 0.001660266666666151, "sum": 2.229065799999148}, "parse_seconds": {"mean": 0.0004646215112316196, "median": 0.00015539999998281928, "p95": 0.00268783500001746, "sum": 1.3650579999984984}, "parsing_failure_rate": 0.14805990469707284, "unsupported_rate": 0.20626276378488767}, "improved_final": {"n": 2938, "correct": 2525, "accuracy": 85.9428182437032, "precision": 0.9696412143514259, "recall": 0.7350069735006973, "f1": 0.8361761205870686, "confusion_matrix": [[1471, 33], [380, 1054]], "labels": {"True": 1434, "False": 1504}, "raw_labels": {"True": 1434, "False": 1394, "Uncertain": 110}, "parsing_failure_count": 435, "unsupported_count": 509, "statuses": {"entailed": 1081, "not_proved": 1331, "unsupported_input": 509, "entailed_partial_input": 6, "resource_limit": 11}, "independent_groups": 320, "error_categories": {"none": 2412, "parsing_failure": 435, "unsupported_rule_representation": 80, "resource_limit": 11}, "inference_seconds": {"mean": 0.00025569104833312743, "median": 1.2900000001536682e-05, "p95": 0.000994050000014113, "sum": 0.7512203000027284}, "response_seconds": {"mean": 0.007414020524162297, "median": 0.006355999999982487, "p95": 0.016851999999971667, "sum": 21.782392299988828}, "amortized_seconds": {"mean": 0.0007033450646695869, "median": 0.0004534875000032912, "p95": 0.0015511999999944237, "sum": 2.0664277999992464}, "parse_seconds": {"mean": 0.0004383601089181059, "median": 0.00015050000001792796, "p95": 0.0025521050000122617, "sum": 1.2879020000013952}, "parsing_failure_rate": 0.14805990469707284, "unsupported_rate": 0.17324710687542547}, "validation_datasets": {"overall": {"baseline": 0.8316679477325134, "improved": 0.8758647194465795, "delta_percentage_points": 4.419677171406611}, "RuleTaker": {"baseline": 0.8215102974828375, "improved": 0.8872997711670481, "delta_percentage_points": 6.578947368421051}, "CounterBench": {"baseline": 0.9732142857142857, "improved": 0.9732142857142857, "delta_percentage_points": 0.0}, "FOLIO": {"baseline": 0.6224489795918368, "improved": 0.6224489795918368, "delta_percentage_points": 0.0}, "multi_hop": {"baseline": 0.7477148080438757, "improved": 0.8592321755027422, "delta_percentage_points": 11.151736745886653}}, "final_datasets": {"overall": {"baseline": 0.7937372362151123, "improved": 0.859428182437032, "delta_percentage_points": 6.569094622191973}, "RuleTaker": {"baseline": 0.8123088685015291, "improved": 0.886085626911315, "delta_percentage_points": 7.3776758409785925}, "FOLIO": {"baseline": 0.6428571428571429, "improved": 0.6428571428571429, "delta_percentage_points": 0.0}, "multi_hop": {"baseline": 0.6917197452229299, "improved": 0.8573248407643312, "delta_percentage_points": 16.560509554140136}}, "final_delta_ci": {"groups": 320, "repetitions": 10000, "ci95_percentage_points": [4.339051463168516, 8.973010865755345], "seed": 20261030}, "diagnostic_accuracy": {"baseline": 100.0, "improved": 100.0}, "local_parser_tradeoff": {"accuracy": 84.47348193697157, "runtime_seconds": 771.7328535000082, "actual_invocation_seconds": 226.74220559999992, "FOLIO_accuracy": 73.80952380952381, "note": "Optional local parser policy on validation: only FOLIO uses translation in this run; RuleTaker and CounterBench use deterministic paths."}}')
for split in ('baseline_validation','improved_validation','baseline_final','improved_final'):
    m=GENERALIZATION_AUDIT[split]
    print(f"{split}: {m['correct']}/{m['n']} accuracy={m['accuracy']:.2f}% "
          f"median response={m['response_seconds']['median']:.4f}s "
          f"p95 response={m['response_seconds']['p95']:.4f}s")
print('Final paired clustered 95% interval (percentage points):', GENERALIZATION_AUDIT['final_delta_ci']['ci95_percentage_points'])
print('Diagnostics:', GENERALIZATION_AUDIT['diagnostic_accuracy'])


baseline_validation: 2164/2602 accuracy=83.17% median response=0.0033s p95 response=0.0078s
improved_validation: 2279/2602 accuracy=87.59% median response=0.0052s p95 response=0.0107s
baseline_final: 2332/2938 accuracy=79.37% median response=0.0070s p95 response=0.0155s
improved_final: 2525/2938 accuracy=85.94% median response=0.0064s p95 response=0.0169s
Final paired clustered 95% interval (percentage points): [4.339051463168516, 8.973010865755345]
Diagnostics: {'baseline': 100.0, 'improved': 100.0}


## Measured public evaluation

Development used jise/train, the first 150 FOLIO training examples, and CounterBench
groups selected by `(question_id % 200) % 5 < 3`. CounterBench held-out groups use the
remaining residues, keeping the five name substitutions of each template together.
The public test data had been examined in the preceding review, so these results
are public benchmark results, not a claim of blind or private-test evaluation.

Every dataset starts with a fresh agent; context caching is enabled within a batch.
Parsing, reasoning, and total batch time are measured separately. The alternative
200-example RuleTaker prefix contains only depth-1 cases.


In [16]:
from dataclasses import asdict
def evaluate(name,rows,formal=False,mode='owa'):
    agent=ReasoningAgent(mode=mode,timeout=0.5)
    predictions=[];t0=time.perf_counter()
    for i,row in enumerate(rows):
        r=agent.predict_record(row,formal_annotations=formal)
        raw=row.get('flag',row.get('label',row.get('answer')))
        gold=binary_label(raw)
        predictions.append(dict(index=i,id=str(row.get('id',row.get('example_id',row.get('question_id',i)))),gold=gold,raw_label=raw,correct=r.prediction==gold,**asdict(r)))
    elapsed=time.perf_counter()-t0
    correct=sum(x['correct'] for x in predictions)
    cm=[[sum(p['gold']==g and p['prediction']==v for p in predictions) for v in (False,True)] for g in (False,True)]
    tn,fp=cm[0];fn,tp=cm[1];precision=tp/(tp+fp) if tp+fp else 0;recall=tp/(tp+fn) if tp+fn else 0
    summary=dict(name=name,n=len(rows),correct=correct,accuracy=100*correct/len(rows) if rows else 0,precision=precision,recall=recall,f1=2*precision*recall/(precision+recall) if precision+recall else 0,confusion_matrix=cm,total_seconds=elapsed,parse_seconds=sum(r['parse_seconds'] for r in predictions),inference_seconds=sum(r['inference_seconds'] for r in predictions),statuses=dict(Counter(r['status'] for r in predictions)),formal_annotations=formal,semantics=mode)
    # Preserve a separately named subset metric for comparison with originals.
    binary=[r for r in predictions if str(r['raw_label']).lower() not in ('uncertain','unknown')]
    summary['true_false_only']={'n':len(binary),'correct':sum(r['correct'] for r in binary),'accuracy':100*sum(r['correct'] for r in binary)/len(binary) if binary else 0}
    print(json.dumps(summary),flush=True)
    return {'summary':summary,'predictions':predictions}
evaluation_results = []
jobs = [
    ('jise_test', PUBLIC_DATA['jise_test'], False, 'owa'),
    ('jise_ood_test', PUBLIC_DATA['jise_ood_test'], False, 'owa'),
    ('tasksource_dev200_cwa', PUBLIC_DATA['tasksource_dev200'], False, 'cwa'),
    ('folio_validation_nl', PUBLIC_DATA['folio_validation'], False, 'owa'),
    ('folio_validation_formal', PUBLIC_DATA['folio_validation'], True, 'owa'),
]
for part in ('alpha', 'backdoor'):
    rows = PUBLIC_DATA['counterbench_' + part]
    heldout = [r for r in rows if int(r['question_id']) % 200 % 5 >= 3]
    jobs.extend([('counterbench_' + part + '_heldout', heldout, False, 'owa'),
                 ('counterbench_' + part + '_all', rows, False, 'owa')])
for name, rows, formal, mode in jobs:
    evaluation_results.append(evaluate(name, rows, formal, mode))
print('\nFINAL ACCURACY AND INFERENCE TIME')
for entry in evaluation_results:
    s = entry['summary']
    print(f"{s['name']:34s} {s['correct']:4d}/{s['n']:<4d} accuracy={s['accuracy']:6.2f}% "
          f"parse={s['parse_seconds']:.4f}s inference={s['inference_seconds']:.4f}s batch={s['total_seconds']:.4f}s")
Path('evaluation_results.json').write_text(json.dumps(evaluation_results, indent=2), encoding='utf-8')


{"name": "jise_test", "n": 250, "correct": 244, "accuracy": 97.6, "precision": 0.9629629629629629, "recall": 1.0, "f1": 0.9811320754716981, "confusion_matrix": [[88, 6], [0, 156]], "total_seconds": 0.420899400000053, "parse_seconds": 0.34298909999688476, "inference_seconds": 0.07291279999753897, "statuses": {"not_proved": 88, "entailed": 162}, "formal_annotations": false, "semantics": "owa", "true_false_only": {"n": 250, "correct": 244, "accuracy": 97.6}}
{"name": "jise_ood_test", "n": 375, "correct": 373, "accuracy": 99.46666666666667, "precision": 0.9849624060150376, "recall": 1.0, "f1": 0.9924242424242424, "confusion_matrix": [[242, 2], [0, 131]], "total_seconds": 0.6023255999998582, "parse_seconds": 0.4822277000039321, "inference_seconds": 0.11254579999240377, "statuses": {"entailed": 133, "not_proved": 242}, "formal_annotations": false, "semantics": "owa", "true_false_only": {"n": 375, "correct": 373, "accuracy": 99.46666666666667}}
{"name": "tasksource_dev200_cwa", "n": 200, "cor

## Local parser study: development, reserved stories and public validation

The local model translates source text into formulas; the Python classical engine
decides entailment. Structural checks preserve source-index coverage, predicate
arity and variable binding. Validator feedback may request a corrected translation.
Checks cannot guarantee that a syntactically valid translation matches the English.

Six development stories contain 21 queries. Sixteen reserved training stories
contain 48 queries and were excluded from the previously examined training prefix.
The parser configuration was frozen before scoring those reserved stories. The
separate public validation set was already inspected in earlier passes.

By default this cell **replays recorded translations**, freshly recomputing every
proof and prediction. It prints the original measured translation duration and
the current replay duration separately. It does not call replay time end-to-end
inference time. Set LIVE_LOCAL_STUDY=True after starting the local server to repeat
translation live. New inputs never use this evaluation translation bundle.


In [17]:
def replay_study(bundle,live=False,endpoint='http://127.0.0.1:8765'):
    """Cached mode measures proof replay, never relabels it as live translation."""
    results=[]
    for phase,data in bundle.items():
        rows=data['rows'];by_story=defaultdict(list)
        for row in rows:by_story[str(row['story_id'])].append(row)
        frozen={}
        for translation in data['translations']:
            group=by_story[translation['story_id']]
            frozen[(group[0]['premises'],tuple(r['conclusion'] for r in group))]=translation
        class FrozenTranslator:
            def translate(self,context,queries):
                item=frozen[(context,tuple(queries))]
                if 'error' in item:raise ValueError(item['error'])
                return item
        agent=LocalReasoningAgent(endpoint=endpoint,translator=None if live else FrozenTranslator())
        start=time.perf_counter();predictions=agent.predict_records(rows);elapsed=time.perf_counter()-start
        baseline=ReasoningAgent();base=[baseline.predict_record(row) for row in rows]
        def score(items):
            gold=[binary_label(row['label']) for row in rows]
            tp=sum(y and r.prediction for y,r in zip(gold,items));fp=sum(not y and r.prediction for y,r in zip(gold,items));fn=sum(y and not r.prediction for y,r in zip(gold,items))
            correct=sum(y==r.prediction for y,r in zip(gold,items))
            return {'n':len(rows),'correct':correct,'accuracy':100*correct/len(rows),
                    'precision':tp/(tp+fp) if tp+fp else 0,'recall':tp/(tp+fn) if tp+fn else 0}
        result={'phase':phase,'mode':'live_translation' if live else 'cached_translation_proof_replay',
                'deterministic':score(base),'local_parser':score(predictions),
                'recorded_translation_seconds':data['translation_seconds'],'replay_or_live_seconds':elapsed,
                'predictions':[{'example_id':row['example_id'],'prediction':r.prediction,'status':r.status,'trace':r.trace} for row,r in zip(rows,predictions)]}
        if not live:
            expected=next(s for s in data['summary'] if s['method']=='neural_ground')
            assert (result['local_parser']['n'],result['local_parser']['correct'])==(expected['n'],expected['correct']), 'Replay differs from measured study'
        print(phase, result['mode'], 'baseline',result['deterministic'],'local parser',result['local_parser'],
              'recorded translation seconds',round(data['translation_seconds'],3),'current execution seconds',round(elapsed,3))
        results.append(result)
    return results
STUDY_B64 = '...'
STUDY_DATA = json.loads(zlib.decompress(base64.b64decode(STUDY_B64)))
LIVE_LOCAL_STUDY = False
study_results = replay_study(STUDY_DATA, live=LIVE_LOCAL_STUDY)
Path('study_replay_results.json').write_text(json.dumps(study_results, indent=2), encoding='utf-8')


dev cached_translation_proof_replay baseline {'n': 21, 'correct': 14, 'accuracy': 66.66666666666667, 'precision': 1.0, 'recall': 0.125} local parser {'n': 21, 'correct': 19, 'accuracy': 90.47619047619048, 'precision': 0.875, 'recall': 0.875} recorded translation seconds 44.13 current execution seconds 0.027
reserved cached_translation_proof_replay baseline {'n': 48, 'correct': 32, 'accuracy': 66.66666666666667, 'precision': 0.6666666666666666, 'recall': 0.11764705882352941} local parser {'n': 48, 'correct': 33, 'accuracy': 68.75, 'precision': 0.6666666666666666, 'recall': 0.23529411764705882} recorded translation seconds 89.892 current execution seconds 0.141
validation cached_translation_proof_replay baseline {'n': 203, 'correct': 142, 'accuracy': 69.95073891625616, 'precision': 0.7619047619047619, 'recall': 0.2222222222222222} local parser {'n': 203, 'correct': 162, 'accuracy': 79.80295566502463, 'precision': 0.8444444444444444, 'recall': 0.5277777777777778} recorded translation seco

## Optional live local parser setup

These helpers download a checksum-verified Qwen3-4B-Instruct-2507 Q4_K_M model
(about 2.5 GB) and a pinned llama.cpp Windows x64 runtime into the user's cache.
CPU and CUDA 12.4 builds are supported. No API key is needed. On other platforms,
run a compatible llama.cpp server on http://127.0.0.1:8765 and use the same model.

Run `setup_local_parser(cuda=True)` and then
`parser_process, parser_endpoint = start_local_parser(cuda=True)` for an NVIDIA GPU.
Use `cuda=False` for CPU. These downloads and startup are **not automatic**.
Stop the server afterward with `parser_process.terminate(); parser_process.wait()`.
The local model is only a semantic parser; the submitted classical code decides.


In [18]:
"""Download a pinned local parser model and a checksum-verified llama.cpp runtime.

Large files live in the user's cache, outside the submission/OneDrive folder.
This script installs no service and makes no global environment changes.
"""
from pathlib import Path
import argparse
import hashlib
import json
import time
import urllib.request
import zipfile

CACHE=Path.home()/'.cache'/'ai-assignment'/'local-parser'
MODEL_NAME='Qwen3-4B-Instruct-2507-Q4_K_M.gguf'
FILES=[
    ('llama-b11970-bin-win-cpu-x64.zip',
     'https://github.com/ggml-org/llama.cpp/releases/download/b11970/llama-b11970-bin-win-cpu-x64.zip',
     '469d0ee06555ed35f846c881708544e0d8787adcb49409571e435b88933daf16',24408564),
    (MODEL_NAME,
     'https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/'+MODEL_NAME,
     '3605803b982cb64aead44f6c1b2ae36e3acdb41d8e46c8a94c6533bc4c67e597',2497281120),
]
CUDA_FILES=[
    ('llama-b11970-bin-win-cuda-12.4-x64.zip',
     'https://github.com/ggml-org/llama.cpp/releases/download/b11970/llama-b11970-bin-win-cuda-12.4-x64.zip',
     'f80904d4639d9a2e01c9ab828e9caffd56ce3ab8f47f08b352ba81d521cf68ec',279764162),
    ('cudart-llama-bin-win-cuda-12.4-x64.zip',
     'https://github.com/ggml-org/llama.cpp/releases/download/b11970/cudart-llama-bin-win-cuda-12.4-x64.zip',
     '8c79a9b226de4b3cacfd1f83d24f962d0773be79f1e7b75c6af4ded7e32ae1d6',391443627),
]

def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def setup_local_parser(cache=CACHE,cuda=False):
    cache=Path(cache).resolve();cache.mkdir(parents=True,exist_ok=True)
    files=([FILES[1]]+CUDA_FILES) if cuda else FILES
    for name,url,checksum,size in files:
        target=cache/name
        if target.exists() and target.stat().st_size==size and digest(target)==checksum:
            print('Verified existing',name,flush=True);continue
        temporary=cache/(name+'.partial');start=time.monotonic();last=start
        request=urllib.request.Request(url,headers={'User-Agent':'AI-assignment-local-parser'})
        with urllib.request.urlopen(request,timeout=60) as response,temporary.open('wb') as output:
            received=0
            while True:
                block=response.read(4*1024*1024)
                if not block:break
                output.write(block);received+=len(block);now=time.monotonic()
                if now-last>10:
                    print(f'{name}: {received/size:.1%}, {received/1e6:.0f} MB, {received/1e6/(now-start):.1f} MB/s',flush=True);last=now
        if temporary.stat().st_size!=size or digest(temporary)!=checksum:raise ValueError('Download integrity check failed: '+name)
        temporary.replace(target);print('Downloaded and verified',name,flush=True)
    runtime=cache/('runtime-b11970-cuda' if cuda else 'runtime-b11970');runtime.mkdir(exist_ok=True)
    for name,_,_,_ in files:
        if not name.endswith('.zip'):continue
        with zipfile.ZipFile(cache/name) as archive:
            for entry in archive.namelist():
                if not (runtime/entry).resolve().is_relative_to(runtime):raise ValueError('Invalid archive path')
            archive.extractall(runtime)
    metadata={'model':MODEL_NAME,'model_sha256':FILES[1][2],'model_source':FILES[1][1],
              'runtime':'llama.cpp b11970 '+('CUDA 12.4' if cuda else 'CPU')+' x64','runtime_files':files,
              'cache':str(cache),'note':'Model translates language only; classical Python code decides entailment.'}
    (cache/('manifest-cuda.json' if cuda else 'manifest.json')).write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    print(json.dumps(metadata,indent=2),flush=True)
    return cache


"""Run the downloaded CPU parser on loopback. Ctrl+C stops this child server."""
from pathlib import Path
import json
import os
import subprocess
import time
import urllib.request

def start_local_parser(cache=None,port=8765,threads=6,cuda=False):
    cache=Path(cache or Path.home()/'.cache'/'ai-assignment'/'local-parser')
    executable=cache/('runtime-b11970-cuda' if cuda else 'runtime-b11970')/'llama-server.exe'
    model=cache/'Qwen3-4B-Instruct-2507-Q4_K_M.gguf'
    if not executable.exists() or not model.exists():raise FileNotFoundError('Run setup_local_parser.py first (Windows x64 CPU build).')
    args=[str(executable),'-m',str(model),'--host','127.0.0.1','--port',str(port),'-c','8192' if cuda else '4096',
          '-t',str(threads),'-tb',str(threads),'-b','256','-ub','128','-ngl','99' if cuda else '0','--parallel','1',
          '--cache-type-k','q8_0','--cache-type-v','q8_0']
    with (cache/'server.log').open('ab') as log:
        process=subprocess.Popen(args,stdout=log,stderr=log,creationflags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0)
    endpoint=f'http://127.0.0.1:{port}'
    try:
        for _ in range(60):
            if process.poll() is not None:raise RuntimeError('Parser exited. Inspect '+str(cache/'server.log'))
            try:
                with urllib.request.urlopen(endpoint+'/health',timeout=1) as response:
                    if json.load(response).get('status')=='ok':return process,endpoint
            except OSError:pass
            time.sleep(1)
        raise TimeoutError('Parser did not become ready within 60 seconds')
    except BaseException:
        process.terminate();process.wait(timeout=15);raise



## Error analysis and limits

The language-only FOLIO result is the appropriate measure when only context/query
are provided. Its broad English coverage remains limited. The formal path supports
quantifiers, conjunction, disjunction, implication, XOR, biconditional and equality,
but may stop at grounding/equality/time budgets. Entire malformed formal premises
can be omitted with an explicit diagnostic; a positive proof from the remaining
subset is still valid, while a failed proof is flagged as incomplete input.

The causal engine assumes a Boolean acyclic SCM, conditions exogenous assignments
on factual observations, and recomputes the intervention world. Unspecified root
priors default to independent Bernoulli(0.5). Probability > 0.5 gives the binary
causal answer, while exact probability is retained. Cycles, incompatible equations,
and impossible observations are reported instead of silently guessed.

Some public labels/annotations disagree with the stated facts or formulas. Their
original labels are kept unchanged for scoring; no case IDs or answer overrides
exist in the reasoning code. No private test set was accessed.


In [19]:

for entry in evaluation_results:
    failed = [r for r in entry['predictions'] if not r['correct']]
    print(entry['summary']['name'], 'statuses:', entry['summary']['statuses'])
    for row in failed[:2]:
        print('  id=', row['id'], 'gold=', row['gold'], 'pred=', row['prediction'],
              'status=', row['status'], 'diagnostics=', row['diagnostics'][:1])


jise_test statuses: {'not_proved': 88, 'entailed': 162}
  id= 10 gold= False pred= True status= entailed diagnostics= []
  id= 111 gold= False pred= True status= entailed diagnostics= []
jise_ood_test statuses: {'entailed': 133, 'not_proved': 242}
  id= 296 gold= False pred= True status= entailed diagnostics= []
  id= 331 gold= False pred= True status= entailed diagnostics= []
tasksource_dev200_cwa statuses: {'entailed': 104, 'not_proved': 96}
folio_validation_nl statuses: {'not_proved': 177, 'entailed': 20, 'unsupported_input': 5, 'entailed_partial_input': 1}
  id= 1015 gold= True pred= False status= not_proved diagnostics= []
  id= 1210 gold= True pred= False status= not_proved diagnostics= []
folio_validation_formal statuses: {'unsupported_input': 37, 'not_proved': 116, 'entailed': 41, 'entailed_partial_input': 2, 'resource_limit': 7}
  id= 1015 gold= True pred= False status= unsupported_input diagnostics= ['Whole premise omitted: Trailing FOL tokens: )']
  id= 57 gold= True pred= F

## Generate submission.csv

Set `INPUT_PATH` to a CSV, JSON list, or JSONL file. Required fields are `context`
and `statement` (aliases `premises`, `given_info`, `question`, `conclusion`, `query`
are supported); `id` is optional. Leave it as None to regenerate the included
public RuleTaker test demonstration CSV. It is **not** a private-test submission.
Use `USE_FORMAL_ANNOTATIONS=True` only if the input genuinely supplies both FOL fields.


In [20]:

INPUT_PATH = None
OUTPUT_PATH = 'submission.csv'
SEMANTICS = 'owa'
USE_FORMAL_ANNOTATIONS = False
PARSER_MODE = 'deterministic'  # Options: deterministic, local, adaptive. Local modes need the server.
input_rows = read_records(INPUT_PATH) if INPUT_PATH else PUBLIC_DATA['jise_test']
submission_agent = (AdaptiveReasoningAgent(mode=SEMANTICS) if PARSER_MODE == 'adaptive' else
                    LocalReasoningAgent(mode=SEMANTICS) if PARSER_MODE == 'local' else ReasoningAgent(mode=SEMANTICS))
csv_start = time.perf_counter()
DIAGNOSTICS_PATH = 'submission_diagnostics.json' if PARSER_MODE in ('local','adaptive') else None
csv_rows = write_submission(input_rows, OUTPUT_PATH, submission_agent, USE_FORMAL_ANNOTATIONS, diagnostics_path=DIAGNOSTICS_PATH)
assert len(csv_rows) == len(input_rows)
assert all(r['predicted_label'] in ('True', 'False') for r in csv_rows)
assert list(csv_rows[0]) == ['id', 'predicted_label'] if csv_rows else True
print(f'Generated {OUTPUT_PATH}: {len(csv_rows)} rows in {time.perf_counter()-csv_start:.4f}s')
print(csv_rows[:5])
if DIAGNOSTICS_PATH:
    evidence = json.loads(Path(DIAGNOSTICS_PATH).read_text(encoding='utf-8'))
    print('Local pipeline statuses:', dict(Counter(r['status'] for r in evidence)))
    print('Translated formulas and proof traces:', DIAGNOSTICS_PATH)


Generated submission.csv: 250 rows in 0.4271s
[{'id': 0, 'predicted_label': 'False'}, {'id': 1, 'predicted_label': 'False'}, {'id': 2, 'predicted_label': 'False'}, {'id': 3, 'predicted_label': 'False'}, {'id': 4, 'predicted_label': 'False'}]
